# WasteVision: training notebook (final run)
**Hazard-aware routing of household waste to India's four SWM Rules 2026 streams, tested under source shift**

One end-to-end run: download every source, clean, de-duplicate across sources, make frozen photo-grouped splits,
train the CNNs (the main model with three seeds), evaluate, run the research analyses and export the model.

| Source | What it is | Used for |
|---|---|---|
| Garbage Dataset v2 (Kaggle) | household photos (mixed sources) | battery, shoes, clothes, food and most dry classes |
| RealWaste (UCI) | real landfill photos | 9 classes |
| TrashNet | photos taken by the authors | 6 classes |
| Drinking Waste (Kaggle) | real photos of cans and bottles | metal, glass, plastic |
| Portland State (Singh) | 11,500 single-item photos (NumPy archive) | boxes, glass bottles, cans, plastic bottles |
| TIDY | original photos | several classes |
| TACO | litter in the wild, segmented | object crops |
| ZeroWaste-f | recycling-plant conveyor, segmented | plastic, cardboard, metal crops |
| BePLi v2 | beach plastic litter, segmented | plastic crops |
| DWSD | Kolkata streets, segmented | Indian test set + training crops |
| TrashBox | web images | medicines/syringes and e-waste folders only |
| Mayank14 (Hugging Face) | 9-class waste photos | adds medicinal and electronic variety |
| GlobalWasteData (DFKI) | 89,807 images from 19 public sets | battery, e-waste, medicines, diapers and more (after de-duplication) |
| Singapore Battery (BatSort) | ~500 battery photos | second battery source |
| MedPackID (Mendeley) | medicine packaging and pills, Indonesia, boxes | medicine crops |
| Pharmaceutical synthetic (Kaggle) | synthetic pill images | training only, capped at 500 |
| Sanitary Pad (conveyor belt, Kaggle) | wrapped and open pads | new sanitary class (fourth stream) |
| WastePro (Kaggle) | compiled solid-waste photos | general classes (after de-duplication) |

Every wanted source must load: a source that still fails after retries stops the run with a clear message.
Near-duplicates are removed across all sources (pHash, distance ≤ 5) and the full pair list is saved for the audit.
Splits are 70 / 10 / 20, grouped by parent photo; synthetic images are training-only.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Allow Google Drive access.
Outputs go to `MyDrive/WasteVision_final`. If Colab disconnects, **Run all** again: finished steps are skipped.


In [ ]:
#@title 1 · Settings (the only cell you may want to edit)
QUICK_TEST = False   # True = small trial run that checks the whole pipeline end to end
RETRAIN = False      # True = train again even if a saved model already exists on Drive

PROJECT_DIR = "/content/drive/MyDrive/WasteVision_final"   # durable outputs (survive disconnects)
WORK_DIR = "/content/wv_work"                             # fast local scratch space

MAIN_ARCH = "efficientnetv2b0"
MAIN_SEEDS = [42, 7, 123]   # the main model is trained three times; results are reported as mean ± sd
MODELS = [MAIN_ARCH, "mobilenetv2"] + [f"{MAIN_ARCH}_seed{s}" for s in MAIN_SEEDS[1:]] + ["scratch_cnn"]

def base_arch(name):
    return name.split("_seed")[0]

def seed_of(name):
    return int(name.split("_seed")[1]) if "_seed" in name else SEED
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

# Data sources (set False to skip one)
USE = dict(gd=True, trashbox=True, dwsd=True, realwaste=True, trashnet=True, drinking=True, psu=True, tidy=True,
           taco=True, zerowaste=True, bepli=True, mayank=True, globalwaste=True, sgbattery=True, medpack=True,
           pharma=True, sanitary=True, wastepro=True)
DOWNLOAD_RETRIES = 4          # every source is retried; the run stops if any wanted source still yields zero images
SYNTH_SOURCES = {"PharmaSynth"}   # synthetic images: training split only, never validation or test
SYNTH_CAP = 500                   # at most this many synthetic images

# Data
CAP_PER_CLASS = 2500          # max images per class (balanced across sources); DWSD is never capped
TEST_FRACTION, VAL_FRACTION = 0.20, 0.10
PHASH_DUP_THRESHOLD = 5       # Hamming distance at or below which two images count as duplicates
DWSD_GROUP_THRESHOLD = 10     # looser threshold to group DWSD photos that are flips/rotations of each other
DWSD_TEST_FRACTION = 0.30     # share of DWSD photo groups held out as the Indian test set
MIN_CROP_SIDE = 40            # object crops smaller than this (pixels) are dropped
MIN_CROP_AREA = 600

# Training
EPOCHS_SCRATCH = 12
EPOCHS_HEAD = 8               # phase 1: backbone frozen
EPOCHS_FINETUNE = 10          # phase 2: last layers unfrozen
UNFREEZE_LAST_LAYERS = 30
LR_SCRATCH, LR_HEAD, LR_FINETUNE = 1e-3, 1e-3, 1e-5
PRETRAINED_WEIGHTS = "imagenet"
BALANCED_BATCHES = True       # equal class shares in every training batch

# Decisions
COST_RATIO = 4.0                          # missing a hazardous item is 4x worse than a false alarm -> threshold 1/(1+4) = 0.20
SENSITIVITY_COST_RATIOS = [2.0, 4.0, 9.0] # thresholds 0.33, 0.20, 0.10
TARGET_STREAM_ACC = 0.97                  # "not sure" threshold: keep stream accuracy on answered items at or above this
TARGET_CLASS_ACC = 0.95
BOOTSTRAP_ROUNDS = 1000

REAL_SOURCES = {"RealWaste", "TrashNet", "Drinking", "PSU", "TIDY", "TACO", "ZeroWaste", "BePLi", "DWSD", "MedPackID", "Sanitary"}
# Analyses added in the final run
LODO_MIN_IMAGES = 150         # a source is held out as its own test domain if it has at least this many images
HOLDOUT_UNKNOWN = ["battery", "shoes"]   # hazardous and harmless classes held out to test the "not sure" answer
DUP_SWEEP = list(range(0, 11))
AUDIT_PAIRS = 200

In [ ]:
#@title 2 · Setup
import os, sys, json, time, shutil, subprocess, hashlib, re, math, random, glob, zipfile, tarfile, warnings, datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.isdir("/kaggle/working")   # Kaggle notebooks: no Drive; outputs go to the notebook's Output tab
if IN_KAGGLE and not os.environ.get("WV_PROJECT_DIR"):
    PROJECT_DIR = "/kaggle/working/WasteVision_final"
    WORK_DIR = "/tmp/wv_work"

# Environment overrides (used only for local testing of this notebook)
if os.environ.get("WV_PROJECT_DIR"): PROJECT_DIR = os.environ["WV_PROJECT_DIR"]
if os.environ.get("WV_WORK_DIR"): WORK_DIR = os.environ["WV_WORK_DIR"]
if os.environ.get("WV_QUICK"): QUICK_TEST = os.environ["WV_QUICK"] == "1"
if os.environ.get("WV_WEIGHTS"):
    PRETRAINED_WEIGHTS = None if os.environ["WV_WEIGHTS"].lower() == "none" else os.environ["WV_WEIGHTS"]
LOCAL_SOURCES = json.loads(os.environ.get("WV_LOCAL_SOURCES", "{}"))

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import imagehash
except ImportError:
    pip_install("imagehash")
    import imagehash

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if QUICK_TEST:
    PROJECT_DIR = PROJECT_DIR.rstrip("/") + "/quick_test"
    WORK_DIR = WORK_DIR.rstrip("/") + "_quick"
    EPOCHS_SCRATCH, EPOCHS_HEAD, EPOCHS_FINETUNE = 2, 1, 1
    BOOTSTRAP_ROUNDS = 200

DATA_DIR = f"{PROJECT_DIR}/data"
MODELS_DIR = f"{PROJECT_DIR}/models"
RESULTS_DIR = f"{PROJECT_DIR}/results"
FIG_DIR = f"{RESULTS_DIR}/figures"
EXPORT_DIR = f"{PROJECT_DIR}/export"
FEEDBACK_DIR = f"{PROJECT_DIR}/feedback"
RAW_DIR = f"{WORK_DIR}/raw"
NORM_DIR = f"{WORK_DIR}/norm"
CACHE_DIR = f"{WORK_DIR}/tfcache"
for d in [DATA_DIR, MODELS_DIR, RESULTS_DIR, FIG_DIR, EXPORT_DIR, f"{FEEDBACK_DIR}/images", RAW_DIR, NORM_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*shuffle=True.*")
warnings.filterwarnings("ignore", message=".*least populated class.*")
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

# ---- The 12 material classes and their SWM Rules 2026 streams ----
CLASSES = ["biological", "paper_cardboard", "plastic", "metal", "glass", "clothes", "shoes",
           "trash", "battery", "medicines_syringes", "e_waste", "sanitary"]
CLASS_DISPLAY = {
    "biological": "Food & garden organics", "paper_cardboard": "Paper & cardboard", "plastic": "Plastic (incl. thermocol)",
    "metal": "Metal", "glass": "Glass", "clothes": "Clothes & textiles", "shoes": "Shoes",
    "trash": "Other dry residue", "battery": "Battery", "medicines_syringes": "Medicines & syringes",
    "e_waste": "E-waste", "sanitary": "Sanitary waste (pads, diapers)",
}
STREAMS = ["Wet", "Dry", "Special Care", "Sanitary"]   # the four SWM Rules 2026 source-segregation streams
STREAM_OF = {c: "Dry" for c in CLASSES}
STREAM_OF["biological"] = "Wet"
for c in ["battery", "medicines_syringes", "e_waste"]:
    STREAM_OF[c] = "Special Care"
STREAM_OF["sanitary"] = "Sanitary"
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
CLASS_STREAM_IDX = np.array([STREAMS.index(STREAM_OF[c]) for c in CLASSES])
STREAM_MATRIX = np.zeros((NUM_CLASSES, len(STREAMS)), np.float32)
STREAM_MATRIX[np.arange(NUM_CLASSES), CLASS_STREAM_IDX] = 1.0
SC = STREAMS.index("Special Care")
SC_THRESHOLD = 1.0 / (1.0 + COST_RATIO)

STATS_PATH = f"{RESULTS_DIR}/stats.json"
STATS = json.load(open(STATS_PATH)) if os.path.exists(STATS_PATH) else {}
def save_stats():
    with open(STATS_PATH, "w") as f:
        json.dump(STATS, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

def stable_id(*parts):
    return hashlib.sha1("|".join(str(p) for p in parts).encode()).hexdigest()[:16]

print(f"Colab: {IN_COLAB} | quick test: {QUICK_TEST} | project folder: {PROJECT_DIR}")

## Step 1 · Get the data
If the prepared images and frozen splits already exist on Drive (from an earlier run), they are restored and
the download and cleaning cells are skipped automatically. The test sets therefore stay identical across runs.

In [ ]:
#@title 3 · Restore prepared data if it exists
SPLITS_CSV = f"{DATA_DIR}/splits.csv"
NORM_TAR = f"{DATA_DIR}/normalized_images.tar"

def safe_extract_tar(tar_path, dest):
    with tarfile.open(tar_path) as t:
        if hasattr(tarfile, "data_filter"):
            t.extractall(dest, filter="data")
        else:
            for m in t.getmembers():
                if m.name.startswith("/") or ".." in Path(m.name).parts:
                    raise RuntimeError(f"Unsafe path in archive: {m.name}")
            t.extractall(dest)

DATA_READY = False
if os.path.exists(SPLITS_CSV) and os.path.exists(NORM_TAR):
    if not any(Path(NORM_DIR).rglob("*.jpg")):
        print("Restoring prepared images from Drive ...")
        safe_extract_tar(NORM_TAR, NORM_DIR)
    DATA_READY = True
    print(f"✓ Prepared data found: {SPLITS_CSV}. Skipping download and cleaning.")
else:
    print("No prepared data yet. The next cells will download and prepare it.")

In [ ]:
#@title 4 · Download every dataset (in parallel, with retries; the run stops if any wanted source yields no images)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif", ".tif", ".tiff"}
GD_SLUG = "sumn2u/garbage-classification-v2"
TRASHBOX_REPO = "https://github.com/nikhilvenkatkumsetty/TrashBox"
TRASHBOX_SUBDIR = "TrashBox_train_dataset_subfolders"
DWSD_URL = ("https://data.mendeley.com/public-files/datasets/gr99ny6b8p/files/"
            "d0a8df4f-a6d5-447a-95f9-882c7b8fb896/file_downloaded")
DWSD_SHA256 = "fb9229c330ed253f4fb7c18bd46dddebe4df6f8d3f602747f505ff7601a11737"
REALWASTE_URL = "https://archive.ics.uci.edu/static/public/908/realwaste.zip"
PSU_URL = "https://web.cecs.pdx.edu/~singh/rcyc-web/recycle_data_shuffled.tar.gz"
ZEROWASTE_URL = "https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1"
ZEROWASTE_PASSWORD = "UP#1VuX409z4"   # published on the dataset's Zenodo page (record 4899926)
BEPLI_URL = "https://www.seanoe.org/data/00858/96963/data/106157.tar.gz"
KAGGLE = dict(drinking="arkadiyhacks/drinking-waste-classification",
              trashnet="asdasdasasdas/garbage-classification",
              taco="kneroma/tacotrashdataset")

def run(cmd, **kw):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.run(cmd, check=True, **kw)

def find_dir(root, name):
    for r, ds, _ in os.walk(root):
        for d in ds:
            if d.lower() == name.lower():
                return os.path.join(r, d)
    return None

def http_open(url, timeout=120):
    import urllib.request
    return urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=timeout)

class TooSlow(Exception):
    pass

def download(url, dst, budget_min=None, retries=3):
    if os.path.exists(dst):
        return dst
    tmp = dst + ".part"
    for attempt in range(retries):
        try:   # curl resumes interrupted transfers (Zenodo refuses wget's requests); urllib is the fallback
            if shutil.which("curl"):
                subprocess.run(["curl", "-sSfL", "--retry", "10", "--retry-delay", "15", "--connect-timeout", "60",
                                "-C", "-", "-o", tmp, url], check=True)
            else:
                with http_open(url) as r, open(tmp, "wb") as f:
                    shutil.copyfileobj(r, f, 1 << 22)
            if os.path.getsize(tmp) > 0:
                os.replace(tmp, dst)
                return dst
        except Exception as e:
            print(f"  download attempt {attempt + 1} failed for {url[:80]}: {e}")
            time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"download failed after {retries} attempts: {url}")

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_extract_zip(zip_path, dest, password=None):
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            p = Path(m)
            if p.is_absolute() or ".." in p.parts:
                raise RuntimeError(f"Unsafe path in zip: {m}")
        if password is None:
            z.extractall(dest)
            return
    # password-protected archives: the unzip tool is far faster than Python for large files
    try:
        run(["unzip", "-q", "-o", "-P", password, zip_path, "-d", dest])
    except Exception:
        if not shutil.which("7z"):
            subprocess.run("apt-get -qq install -y p7zip-full > /dev/null", shell=True)
        run(["7z", "x", "-y", f"-p{password}", f"-o{dest}", zip_path], stdout=subprocess.DEVNULL)

def kaggle_download(slug):
    try:
        import kagglehub
    except ImportError:
        pip_install("kagglehub"); import kagglehub
    if IN_COLAB:   # optional: Kaggle credentials stored as Colab secrets
        try:
            from google.colab import userdata
            for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
                try:
                    os.environ.setdefault(k, userdata.get(k))
                except Exception:
                    pass
        except Exception:
            pass
    return kagglehub.dataset_download(slug)

def git_clone(url, dest, sparse=None):
    if os.path.isdir(dest) and any(Path(dest).iterdir()):
        return dest
    if sparse:
        run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", url, dest])
        run(["git", "-C", dest, "sparse-checkout", "set", sparse])
    else:
        run(["git", "clone", "--depth", "1", url, dest])
    return dest

# ---- one function per source; each returns the folder to read ----
def get_gd():
    if LOCAL_SOURCES.get("gd"): return LOCAL_SOURCES["gd"]
    path = kaggle_download(GD_SLUG)
    d = find_dir(path, "standardized_256") or find_dir(path, "original")
    if d is None: raise RuntimeError(f"Could not find the image folders inside {path}")
    return d

def get_trashbox():
    if LOCAL_SOURCES.get("trashbox"): return LOCAL_SOURCES["trashbox"]
    dest = f"{RAW_DIR}/trashbox"
    target = f"{dest}/{TRASHBOX_SUBDIR}"
    if not os.path.isdir(target) or sum(1 for _ in Path(target).rglob("*.jpg")) < 1000:
        shutil.rmtree(dest, ignore_errors=True)
        # only the two folders we still need from this web-scraped set
        run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", TRASHBOX_REPO, dest])
        run(["git", "-C", dest, "sparse-checkout", "set", f"{TRASHBOX_SUBDIR}/medical", f"{TRASHBOX_SUBDIR}/e-waste"])
    return target

def get_dwsd():
    zip_path = LOCAL_SOURCES.get("dwsd_zip") or f"{RAW_DIR}/DSWD.zip"
    manual = f"{PROJECT_DIR}/raw/DSWD.zip"
    if not os.path.exists(zip_path) and os.path.exists(manual):
        zip_path = manual
    if not os.path.exists(zip_path):
        download(DWSD_URL, zip_path)
    STATS["dwsd_zip_sha256_matches"] = sha256_of(zip_path) == DWSD_SHA256
    out = f"{RAW_DIR}/dwsd"
    if not os.path.isdir(out) or not any(Path(out).rglob("*.*")):
        os.makedirs(out, exist_ok=True)
        safe_extract_zip(zip_path, out)
        for inner in list(Path(out).rglob("*.zip")):
            safe_extract_zip(str(inner), str(inner.parent / inner.stem))
    return out

def get_realwaste():
    if LOCAL_SOURCES.get("realwaste"): return LOCAL_SOURCES["realwaste"]
    out = f"{RAW_DIR}/realwaste"
    if not any(Path(out).rglob("*.jpg")) if os.path.isdir(out) else True:
        z = download(REALWASTE_URL, f"{RAW_DIR}/realwaste.zip")
        safe_extract_zip(z, out)
        for inner in list(Path(out).rglob("*.zip")):
            safe_extract_zip(str(inner), str(inner.parent))
        os.remove(z)
    return out

def get_trashnet():
    if LOCAL_SOURCES.get("trashnet"): return LOCAL_SOURCES["trashnet"]
    try:
        return kaggle_download(KAGGLE["trashnet"])
    except Exception as e:
        print("  TrashNet from Kaggle failed, trying GitHub:", e)
        dest = git_clone("https://github.com/garythung/trashnet", f"{RAW_DIR}/trashnet_git")
        for z in Path(dest).rglob("*.zip"):
            safe_extract_zip(str(z), f"{RAW_DIR}/trashnet")
        return f"{RAW_DIR}/trashnet"

def get_drinking():
    if LOCAL_SOURCES.get("drinking"): return LOCAL_SOURCES["drinking"]
    return kaggle_download(KAGGLE["drinking"])

def get_tidy():
    if LOCAL_SOURCES.get("tidy"): return LOCAL_SOURCES["tidy"]
    return git_clone("https://github.com/gale31/TIDY", f"{RAW_DIR}/tidy")

def get_taco():
    if LOCAL_SOURCES.get("taco"): return LOCAL_SOURCES["taco"]
    try:
        return kaggle_download(KAGGLE["taco"])
    except Exception as e:
        print("  TACO from Kaggle failed, using GitHub annotations + original image links:", e)
        dest = git_clone("https://github.com/pedropro/TACO", f"{RAW_DIR}/taco")
        ann = json.load(open(f"{dest}/data/annotations.json"))
        def fetch(img):
            p = f"{dest}/data/{img['file_name']}"
            if os.path.exists(p): return
            os.makedirs(os.path.dirname(p), exist_ok=True)
            try:
                download(img.get("flickr_url") or img["coco_url"], p)
            except Exception:
                pass
        with ThreadPoolExecutor(16) as ex:
            list(ex.map(fetch, ann["images"]))
        return dest

def get_zerowaste():
    if LOCAL_SOURCES.get("zerowaste"): return LOCAL_SOURCES["zerowaste"]
    out = f"{RAW_DIR}/zerowaste"
    if not (os.path.isdir(out) and any(Path(out).rglob("*.json"))):
        z = download(ZEROWASTE_URL, f"{RAW_DIR}/zerowaste-f-final.zip")
        safe_extract_zip(z, out, password=ZEROWASTE_PASSWORD)
        os.remove(z)
    return out

def get_bepli():
    if LOCAL_SOURCES.get("bepli"): return LOCAL_SOURCES["bepli"]
    out = f"{RAW_DIR}/bepli"
    if not (os.path.isdir(out) and any(Path(out).rglob("*.json"))):
        os.makedirs(out, exist_ok=True)
        tgz = download(BEPLI_URL, f"{RAW_DIR}/bepli.tar.gz")
        safe_extract_tar(tgz, out)
        os.remove(tgz)
    return out

PSU_NPZ_LABELS = {0: "boxes", 1: "glass_bottles", 2: "cans", 3: "crushed_cans", 4: "water_bottles"}  # checked visually

def get_psu():
    """The 'high-resolution' Portland State archive holds one NumPy file (x_train/y_train/x_test/y_test,
    128x128 RGB, 2,300 photos per class). Labels are integers; the mapping above was checked against the photos."""
    if LOCAL_SOURCES.get("psu"): return LOCAL_SOURCES["psu"]
    out = f"{RAW_DIR}/psu"
    if os.path.isdir(out) and len(list(Path(out).rglob("*.jpg"))) >= 11000:
        return out
    tgz = download(PSU_URL, f"{RAW_DIR}/psu.tar.gz")
    import io
    with tarfile.open(tgz) as t:
        m = next(m for m in t.getmembers() if m.name.endswith(".npz"))
        data = np.load(io.BytesIO(t.extractfile(m).read()))
    for part in ("train", "test"):
        x, y = data[f"x_{part}"], data[f"y_{part}"].reshape(-1)
        for i, (img, lab) in enumerate(zip(x, y)):
            d = f"{out}/{PSU_NPZ_LABELS[int(lab)]}"
            os.makedirs(d, exist_ok=True)
            Image.fromarray(img.astype(np.uint8)).save(f"{d}/{part}_{i:05d}.jpg", quality=95)
    os.remove(tgz)
    return out

MAYANK_URL = "https://huggingface.co/datasets/Mayank14/Multi-Class-Waste-Image-Classification-Dataset/resolve/main/WasteSnap_Dataset.zip"
GWD_URL = "https://cloud.dfki.de/owncloud/index.php/s/656dTBGXKQBaPPB/download"   # GlobalWasteData (DFKI public share)
SGBATTERY_REPO = "https://github.com/FriedrichZhao/Singapore_Battery_Dataset"
MEDPACK_FILES = {   # Mendeley Data d23kndp3sv v1 (MedPackID), CC BY 4.0
    "images.zip": "34dc80b8-2fe7-43dc-9fd7-89dd1027bc87", "labels.zip": "73cfda9d-232c-4ba0-b457-2687a550cfda",
    "data.yaml": "2d15ad23-1326-4536-a721-6594f9dbbdff", "labels.txt": "5aea9cb9-5e98-4ffa-a686-b9af72e3821a"}
KAGGLE.update(pharma="vencerlanz09/pharmaceutical-drugs-and-vitamins-synthetic-images",
              sanitary="manavdoshi252/sanitary-pad-dataset-conveyor-belt",
              wastepro="namanjain001/comprehensive-solid-waste-image-dataset")

def unzip_to(zip_path, out, remove=True):
    os.makedirs(out, exist_ok=True)
    try:
        run(["unzip", "-q", "-o", zip_path, "-d", out])
    except Exception:
        safe_extract_zip(zip_path, out)
    for inner in list(Path(out).rglob("*.zip")):
        unzip_to(str(inner), str(inner.parent / inner.stem), remove=True)
    if remove:
        os.remove(zip_path)
    return out

def has_images(d, n=1):
    return os.path.isdir(d) and sum(1 for p in Path(d).rglob("*") if p.suffix.lower() in IMG_EXTS) >= n

def get_mayank():
    if LOCAL_SOURCES.get("mayank"): return LOCAL_SOURCES["mayank"]
    out = f"{RAW_DIR}/mayank"
    if not has_images(out, 1000):
        unzip_to(download(MAYANK_URL, f"{RAW_DIR}/mayank.zip"), out)
    return out

def get_globalwaste():
    if LOCAL_SOURCES.get("globalwaste"): return LOCAL_SOURCES["globalwaste"]
    out = f"{RAW_DIR}/globalwaste"
    manual = f"{PROJECT_DIR}/raw/GlobalWasteData.zip"   # fallback: download the share by hand and put the zip here
    if not has_images(out, 10000):
        z = manual if os.path.exists(manual) else download(GWD_URL, f"{RAW_DIR}/globalwaste.zip", retries=5)
        unzip_to(z, out, remove=(z != manual))
    return out

def get_sgbattery():
    if LOCAL_SOURCES.get("sgbattery"): return LOCAL_SOURCES["sgbattery"]
    return git_clone(SGBATTERY_REPO, f"{RAW_DIR}/sgbattery")

def mendeley_file(dataset, file_id, dst):
    return download(f"https://data.mendeley.com/public-files/datasets/{dataset}/files/{file_id}/file_downloaded", dst)

def get_medpack():
    if LOCAL_SOURCES.get("medpack"): return LOCAL_SOURCES["medpack"]
    out = f"{RAW_DIR}/medpack"
    if not has_images(out, 1000):
        os.makedirs(out, exist_ok=True)
        for fn, fid in MEDPACK_FILES.items():
            path = mendeley_file("d23kndp3sv", fid, f"{out}/{fn}")
            if fn.endswith(".zip"):
                unzip_to(path, f"{out}/{fn[:-4]}")
    return out

def get_pharma():
    if LOCAL_SOURCES.get("pharma"): return LOCAL_SOURCES["pharma"]
    return kaggle_download(KAGGLE["pharma"])

def get_sanitary():
    if LOCAL_SOURCES.get("sanitary"): return LOCAL_SOURCES["sanitary"]
    return kaggle_download(KAGGLE["sanitary"])

def get_wastepro():
    if LOCAL_SOURCES.get("wastepro"): return LOCAL_SOURCES["wastepro"]
    return kaggle_download(KAGGLE["wastepro"])


GETTERS = dict(gd=get_gd, trashbox=get_trashbox, dwsd=get_dwsd, realwaste=get_realwaste, trashnet=get_trashnet,
               drinking=get_drinking, psu=get_psu, tidy=get_tidy, taco=get_taco, zerowaste=get_zerowaste,
               bepli=get_bepli, mayank=get_mayank, globalwaste=get_globalwaste, sgbattery=get_sgbattery,
               medpack=get_medpack, pharma=get_pharma, sanitary=get_sanitary, wastepro=get_wastepro)
SOURCES = {}
if not DATA_READY:
    def _get(name):
        t0, err = time.time(), None
        for attempt in range(DOWNLOAD_RETRIES):
            try:
                path = GETTERS[name]()
                n = sum(1 for p in Path(path).rglob("*") if p.suffix.lower() in IMG_EXTS)
                if n == 0:
                    raise RuntimeError(f"no images found under {path}")
                return name, path, None, time.time() - t0
            except Exception as e:
                err = f"{type(e).__name__}: {e}"[:400]
                print(f"  {name}: attempt {attempt + 1}/{DOWNLOAD_RETRIES} failed: {err}")
                time.sleep(15 * (attempt + 1))
        return name, None, err, time.time() - t0
    wanted = [n for n, on in USE.items() if on and (not QUICK_TEST or n in LOCAL_SOURCES or n in ("gd", "trashbox", "dwsd"))]
    with ThreadPoolExecutor(max_workers=len(wanted)) as ex:
        for name, path, err, secs in ex.map(_get, wanted):
            if path:
                SOURCES[name] = path
                print(f"✓ {name:12s} {secs / 60:5.1f} min  {path}")
            else:
                print(f"✗ {name:12s} {secs / 60:5.1f} min  FAILED: {err}")
    STATS["sources_ok"] = sorted(SOURCES)
    STATS["sources_failed"] = sorted(set(wanted) - set(SOURCES))
    save_stats()
    if STATS["sources_failed"]:
        raise RuntimeError(f"These sources failed after {DOWNLOAD_RETRIES} attempts: {STATS['sources_failed']}. "
                           "Fix them and run this cell again; finished downloads are reused.")


In [ ]:
#@title 5 · Map every source to the 12 classes (folder datasets, box-annotated and segmented datasets)

def list_images(root):
    return sorted(str(p) for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

GD_MAP = {"battery": "battery", "biological": "biological", "cardboard": "paper_cardboard", "clothes": "clothes",
          "glass": "glass", "metal": "metal", "paper": "paper_cardboard", "plastic": "plastic",
          "shoes": "shoes", "trash": "trash"}

def trashbox_label(top, sub):
    """v2 keeps only the two TrashBox folders no real-photo dataset covers."""
    top, sub = top.lower().strip(), (sub or "").lower().strip()
    if top == "e-waste":
        return "e_waste"
    if top == "medical":
        return "medicines_syringes" if sub in ("medicines", "syringe", "syringes") else None
    return None

def keyword_label(text):
    """Folder or category name -> our class, by keyword. Order matters (most specific first)."""
    s = re.sub(r"[^a-z0-9]+", " ", text.lower())
    rules = [
        (("cigarette", "glove", "gloove", "rope", "string", "fishing", "buoy", "mask", "unlabeled", "unknown", "blister"), None),
        (("sanitary", "pad", "pads", "napkin", "diaper", "diapers", "tampon"), "sanitary"),
        (("battery", "batteries"), "battery"),
        (("syringe", "medicine", "medical", "pill"), "medicines_syringes"),
        (("e waste", "ewaste", "electronic", "electrical", "cable", "phone", "laptop"), "e_waste"),
        (("shoe", "footwear"), "shoes"),
        (("textile", "cloth", "fabric"), "clothes"),
        (("food can",), "metal"),
        (("food container", "styrofoam", "styrene", "foam", "thermocol"), "plastic"),
        (("food", "organic", "vegetation", "biological", "fruit", "vegetable"), "biological"),
        (("glass",), "glass"),
        (("aluminium", "aluminum", "alucan", "metal", "can", "tin", "pop tab", "aerosol", "scrap"), "metal"),
        (("plastified",), "plastic"),
        (("cardboard", "carton", "box", "paper", "magazine", "tissue", "toilet tube", "pizza"), "paper_cardboard"),
        (("plastic", "pet", "hdpe", "bottle", "bag", "wrapper", "film", "container", "cup", "lid", "straw",
          "utensil", "tube", "packet", "tub", "tupperware", "six pack", "fragment", "case"), "plastic"),
        (("trash", "misc", "residual", "other"), "trash"),
    ]
    words = f" {s} "
    for keys, label in rules:
        if any(f" {k} " in words or (len(k) > 4 and k in s) for k in keys):
            return label
    return None

REALWASTE_MAP = {"cardboard": "paper_cardboard", "food organics": "biological", "glass": "glass", "metal": "metal",
                 "miscellaneous trash": "trash", "paper": "paper_cardboard", "plastic": "plastic",
                 "textile trash": "clothes", "vegetation": "biological"}
TRASHNET_MAP = {"cardboard": "paper_cardboard", "paper": "paper_cardboard", "glass": "glass", "metal": "metal",
                "plastic": "plastic", "trash": "trash"}
DRINKING_MAP = {"alucan": "metal", "glass": "glass", "hdpem": "plastic", "pet": "plastic"}
PSU_MAP = {"boxes": "paper_cardboard", "glass_bottles": "glass", "cans": "metal", "crushed_cans": "metal", "water_bottles": "plastic"}
MAYANK_MAP = {"plastic waste": "plastic", "paper and cardboard waste": "paper_cardboard", "glass waste": "glass",
              "metal waste": "metal", "organic waste": "biological", "electronic waste": "e_waste",
              "medicinal waste": "medicines_syringes", "general waste": "trash",
              "hazardous waste": None}   # mixed hazardous items: no single class, so excluded
# GlobalWasteData sub-classes (the folder nearest the image decides; None = excluded on purpose)
GWD_MAP = {"battery": "battery",
           "electrical_cables": "e_waste", "electronic_chips": "e_waste", "electronic_components": "e_waste",
           "laptops": "e_waste", "small_appliances": "e_waste", "smartphones": "e_waste",
           "brown_glass": "glass", "glass": "glass", "glass_materials": "glass", "green_glass": "glass", "white_glass": "glass",
           "medicine_and_strip": "medicines_syringes", "syringe": "medicines_syringes",
           "diaper": "sanitary", "bandai": "sanitary",
           "gloves": None, "masks": None, "pesticide_bottle": None, "thermometer": None,
           "aluminium": "metal", "beverage_cans": "metal", "metal": "metal", "metal_containers": "metal",
           "metallic_materials": "metal", "spray_cans": None,
           "cigarette_butt": None, "garbage": "trash", "trash": "trash",
           "animal_dead_body": None, "compost": "biological", "foods": "biological", "organic_wastes": "biological",
           "vegetation": "biological", "others": None,
           "cardboard": "paper_cardboard", "carton": "paper_cardboard", "newspaper": "paper_cardboard",
           "paper_cups": "paper_cardboard", "papers": "paper_cardboard", "tetra_pak": "paper_cardboard",
           "box_shaped_case": "plastic", "fragment": "plastic", "other_bottle": "plastic", "other_container": "plastic",
           "pet_bottle": "plastic", "plastic": "plastic", "plastic_bag": "plastic", "plastic_bottle": "plastic",
           "plastic_container": "plastic", "plastic_cups": "plastic", "plastic_litter": "plastic",
           "plastic_materials": "plastic", "styrene_foam": "plastic",
           "buoy": None, "fishing_net": None, "other_fishing_gear": None, "other_string": None, "rope": None,
           "rubber_materials": None, "clothes": "clothes", "shoes": "shoes", "textile_trash": "clothes",
           "cement_based_materials": None, "ceramic_materials": None, "construction_scrap": None, "porcelin": None,
           "wooden_materials": None}
GWD_MAIN = {"battery_energy_waste": "battery", "electronic": "e_waste", "glass": "glass", "metal": "metal",
            "organic_biodegradable": "biological", "paper_cardboard": "paper_cardboard", "plastics": "plastic"}
ZEROWASTE_MAP = {"rigid_plastic": "plastic", "soft_plastic": "plastic", "cardboard": "paper_cardboard", "metal": "metal"}
BEPLI_MAP = {"pet_bottle": "plastic", "other_bottle": "plastic", "plastic_bag": "plastic", "box_shaped_case": "plastic",
             "other_container": "plastic", "styrene_foam": "plastic", "fragment": "plastic"}

def folder_rows(source, root, mapper, depth_label=None):
    """Classification datasets: the class is a folder name somewhere in the path."""
    rows = []
    for p in list_images(root):
        rel = os.path.relpath(p, root)
        parts = [x for x in Path(rel).parts[:-1]]
        label, orig = None, None
        for part in reversed(parts):
            key = part.lower().strip()
            lab = mapper(key) if callable(mapper) else mapper.get(key)
            if lab is not None or (not callable(mapper) and key in mapper):
                label, orig = lab, part
                break
        if orig is None and parts:
            orig = parts[-1]
        rows.append(dict(source=source, src_path=p, rel=rel, orig_class=orig or "?", label=label, bbox=None, img_wh=None))
    return rows

def coco_rows(source, root, cat_mapper, group_key=None):
    """Segmentation/detection datasets in COCO format: one crop per annotated object (bbox padded by 10%)."""
    rows = []
    jsons = [p for p in Path(root).rglob("*.json") if p.stat().st_size > 1000]
    index = None
    for jp in jsons:
        try:
            d = json.load(open(jp))
        except Exception:
            continue
        if not isinstance(d, dict) or not {"images", "annotations", "categories"} <= set(d):
            continue
        cats = {c["id"]: c for c in d["categories"]}
        imgs = {i["id"]: i for i in d["images"]}
        for a in d["annotations"]:
            img = imgs.get(a["image_id"])
            c = cats.get(a["category_id"])
            if img is None or c is None or not a.get("bbox"):
                continue
            label = cat_mapper(c)
            x, y, w, h = a["bbox"]
            if label is None or min(w, h) < MIN_CROP_SIDE or w * h < MIN_CROP_AREA:
                continue
            path = jp.parent / img["file_name"]
            if not path.exists():
                for base in (jp.parent / "data", jp.parent / "images", jp.parent.parent / "data", jp.parent.parent):
                    if (base / img["file_name"]).exists():
                        path = base / img["file_name"]; break
            if not path.exists():
                if index is None:
                    index = {Path(p).name: p for p in list_images(root)}
                hit = index.get(Path(img["file_name"]).name)
                if hit is None:
                    continue
                path = Path(hit)
            W, H = img.get("width"), img.get("height")
            px, py = 0.1 * w, 0.1 * h
            bbox = (max(0, x - px), max(0, y - py), x + w + px if not W else min(W, x + w + px),
                    y + h + py if not H else min(H, y + h + py))
            rel = f"{os.path.relpath(path, root)}#{a['id']}"
            parent = os.path.relpath(path, root)
            grp = group_key(parent) if group_key else parent
            rows.append(dict(source=source, src_path=str(path), rel=rel, orig_class=c.get("name", "?"), label=label,
                             bbox=tuple(round(v) for v in bbox), img_wh=(W, H) if W and H else None,
                             group=f"{source}_{grp}"))
    return rows

def yolo_rows(source, root, class_names, mapper):
    """Box-annotated datasets in YOLO format (class cx cy w h, normalised): one padded crop per box."""
    rows = []
    labels = {p.stem: p for p in Path(root).rglob("*.txt") if p.parent.name.lower() != "classes" and p.name != "labels.txt"}
    for ip in list_images(root):
        lp = labels.get(Path(ip).stem)
        if lp is None:
            continue
        try:
            with Image.open(ip) as im:
                W, H = ImageOps.exif_transpose(im).size if im.getexif().get(274, 1) in (5, 6, 7, 8) else im.size
        except Exception:
            continue
        for k, line in enumerate(open(lp).read().split("\n")):
            v = line.split()
            if len(v) < 5:
                continue
            c, (cx, cy, w, h) = int(float(v[0])), map(float, v[1:5])
            label = mapper(class_names[c] if c < len(class_names) else str(c))
            w, h = w * W, h * H
            if label is None or min(w, h) < MIN_CROP_SIDE or w * h < MIN_CROP_AREA:
                continue
            x0, y0 = cx * W - w / 2, cy * H - h / 2
            px, py = 0.1 * w, 0.1 * h
            bbox = (max(0, x0 - px), max(0, y0 - py), min(W, x0 + w + px), min(H, y0 + h + py))
            rel = f"{os.path.relpath(ip, root)}#{k}"
            rows.append(dict(source=source, src_path=ip, rel=rel, orig_class=class_names[c] if c < len(class_names) else str(c),
                             label=label, bbox=tuple(round(x) for x in bbox), img_wh=(W, H),
                             group=f"{source}_{os.path.relpath(ip, root)}"))
    return rows

def roboflow_group(rel):
    """Roboflow exports hold augmented copies named <original>_jpg.rf.<hash>.jpg: keep copies of one photo together."""
    return re.sub(r"\.rf\.[0-9a-f]+$", "", Path(rel).stem)

def taco_label(c):
    name, sup = c.get("name", ""), c.get("supercategory", "")
    if "glass" in name.lower():
        return "glass"
    return keyword_label(name) or keyword_label(sup)

def video_group(rel):
    """Frames from one recording session are near-copies: keep them on one side of the split."""
    stem = Path(rel).stem
    m = re.match(r"(.*?)[_-]?(?:frame)?[_-]?\d+$", stem)
    return f"{Path(rel).parent}/{m.group(1) if m and m.group(1) else stem}"

def build_manifest():
    rows = []
    if "gd" in SOURCES:
        for r in folder_rows("GD", SOURCES["gd"], lambda k: GD_MAP.get(k)):
            rows.append(r)
    if "trashbox" in SOURCES:
        tb = SOURCES["trashbox"]
        for p in list_images(tb):
            rel = os.path.relpath(p, tb); parts = rel.split(os.sep)
            top, sub = parts[0], parts[1] if len(parts) > 2 else None
            rows.append(dict(source="TrashBox", src_path=p, rel=rel, orig_class=f"{top}/{sub}" if sub else top,
                             label=trashbox_label(top, sub), bbox=None, img_wh=None))
    if "realwaste" in SOURCES:
        rows += folder_rows("RealWaste", SOURCES["realwaste"], lambda k: REALWASTE_MAP.get(k))
    if "trashnet" in SOURCES:
        rows += folder_rows("TrashNet", SOURCES["trashnet"], lambda k: TRASHNET_MAP.get(k))
    if "drinking" in SOURCES:
        rows += folder_rows("Drinking", SOURCES["drinking"], lambda k: DRINKING_MAP.get(k.replace(" ", "")))
    if "psu" in SOURCES:
        rows += folder_rows("PSU", SOURCES["psu"], lambda k: PSU_MAP.get(k))
    if "tidy" in SOURCES:
        rows += folder_rows("TIDY", SOURCES["tidy"], keyword_label)
    if "taco" in SOURCES:
        rows += coco_rows("TACO", SOURCES["taco"], taco_label)
    if "zerowaste" in SOURCES:
        rows += coco_rows("ZeroWaste", SOURCES["zerowaste"],
                          lambda c: ZEROWASTE_MAP.get(c.get("name", "").lower().replace(" ", "_")), video_group)
    if "bepli" in SOURCES:
        rows += coco_rows("BePLi", SOURCES["bepli"], lambda c: BEPLI_MAP.get(c.get("name", "").lower()))
    if "mayank" in SOURCES:
        rows += folder_rows("Mayank14", SOURCES["mayank"], MAYANK_MAP)
    if "globalwaste" in SOURCES:
        gw = folder_rows("GlobalWaste", SOURCES["globalwaste"], GWD_MAP)
        for r in gw:   # images directly inside a main-category folder (no sub-class folder)
            if r["label"] is None and r["orig_class"].lower() in GWD_MAIN and r["orig_class"].lower() not in GWD_MAP:
                r["label"] = GWD_MAIN[r["orig_class"].lower()]
        rows += gw
    if "sgbattery" in SOURCES:
        rows += folder_rows("SGBattery", SOURCES["sgbattery"], lambda k: "battery")
    if "medpack" in SOURCES:
        root = SOURCES["medpack"]
        names = [l.strip() for l in open(f"{root}/labels.txt").read().splitlines() if l.strip()] \
            if os.path.exists(f"{root}/labels.txt") else ["medicine_packaging", "pill"]
        rows += yolo_rows("MedPackID", root, names, lambda n: "medicines_syringes")
    if "pharma" in SOURCES:
        ph = pd.DataFrame(folder_rows("PharmaSynth", SOURCES["pharma"], lambda k: None))
        ph = ph[ph.rel.str.contains("Data Combined", regex=False)] if ph.rel.str.contains("Data Combined", regex=False).any() else ph
        per = math.ceil(SYNTH_CAP / max(1, ph.orig_class.nunique()))
        ph = ph.sample(frac=1, random_state=SEED)
        ph = ph[ph.groupby("orig_class").cumcount() < per]
        ph["label"] = "medicines_syringes"
        rows += ph.head(SYNTH_CAP).to_dict("records")
    if "sanitary" in SOURCES:
        san = folder_rows("Sanitary", SOURCES["sanitary"], lambda k: "sanitary" if ("pad" in k or "sanitary" in k) else None)
        for r in san:
            r["group"] = f"Sanitary_{roboflow_group(r['rel'])}"
        rows += san
    if "wastepro" in SOURCES:
        rows += folder_rows("WastePro", SOURCES["wastepro"], lambda k: keyword_label(k.replace("_", " ")))
    df = pd.DataFrame(rows)
    df["id"] = [stable_id(s, r) for s, r in zip(df.source, df.rel)]
    if "group" not in df:
        df["group"] = None
    df["group"] = df.group.where(df.group.notna(), df.source + "_" + df.id)
    # Folder -> class table for every source (checked by eye before training)
    MAP_TABLE = df.groupby(["source", "orig_class"], dropna=False).agg(images=("rel", "size"), label=("label", "first"))
    MAP_TABLE.to_csv(f"{RESULTS_DIR}/label_mapping_by_source.csv")
    with pd.option_context("display.max_rows", 500):
        display(MAP_TABLE)
    excluded = df[df.label.isna()].groupby(["source", "orig_class"]).size()
    STATS["excluded_folders"] = {f"{a}:{b}": int(n) for (a, b), n in excluded.items() if n >= 20}
    STATS["raw_counts"] = {k: int(v) for k, v in df.groupby("source").size().items()}
    df = df[df.label.notna()].copy()
    empty = sorted({ {"gd": "GD", "trashbox": "TrashBox", "realwaste": "RealWaste", "trashnet": "TrashNet", "drinking": "Drinking",
                      "psu": "PSU", "tidy": "TIDY", "taco": "TACO", "zerowaste": "ZeroWaste", "bepli": "BePLi",
                      "mayank": "Mayank14", "globalwaste": "GlobalWaste", "sgbattery": "SGBattery", "medpack": "MedPackID",
                      "pharma": "PharmaSynth", "sanitary": "Sanitary", "wastepro": "WastePro"}[k] for k in SOURCES if k != "dwsd"}
                   - set(df.source))
    if empty:
        raise RuntimeError(f"These sources downloaded but produced no labelled images: {empty}. "
                           "Check the folder -> class table above and fix the mapping.")
    # Avoid wasting time cleaning thousands of images that the per-class cap would drop anyway:
    # keep at most 1.5x the cap per (class, source), sampled by parent photo.
    keep = []
    for (lab, src), g in df.groupby(["label", "source"]):
        if len(g) <= 1.5 * CAP_PER_CLASS:
            keep.append(g); continue
        groups = g.group.drop_duplicates().sample(frac=1, random_state=SEED)
        sizes = g.groupby("group").size().reindex(groups).cumsum()
        keep.append(g[g.group.isin(sizes[sizes <= 1.5 * CAP_PER_CLASS].index)])
    df = pd.concat(keep)
    if QUICK_TEST:
        df = df.sample(frac=1, random_state=SEED).groupby(["source", "label"]).head(8)
    STATS["mapped_counts"] = {f"{a}:{b}": int(n) for (a, b), n in df.groupby(["source", "label"]).size().items()}
    return df.reset_index(drop=True)

if not DATA_READY:
    df_web = build_manifest()
    print("Mapped images per source:", df_web.groupby("source").size().to_dict())
    display(df_web.groupby(["label", "source"]).size().unstack(fill_value=0))

In [ ]:
#@title 6 · DWSD: read masks, cut out one crop per object, group near-identical photos
DWSD_CLASSES = {1: "plastic-container", 2: "plastic-bottle", 3: "thermocol", 4: "metal-bottle",
                5: "plastic-cardboard", 6: "glass", 7: "thermocol-plate", 8: "plastic", 9: "paper",
                10: "plastic-cup", 11: "paper-cup", 12: "aluminium-foil", 13: "cloth", 14: "nylon"}
DWSD_TO_OURS = {"plastic-container": "plastic", "plastic-bottle": "plastic", "thermocol": "plastic",
                "metal-bottle": "metal", "plastic-cardboard": None, "glass": "glass", "thermocol-plate": "plastic",
                "plastic": "plastic", "paper": "paper_cardboard", "plastic-cup": "plastic",
                "paper-cup": "paper_cardboard", "aluminium-foil": "metal", "cloth": "clothes", "nylon": "plastic"}
# Fallback only: class ids ordered by total pixel count in the DWSD paper (Table 3), largest first
DWSD_PIXEL_RANK = [3, 5, 12, 10, 8, 11, 4, 6, 9, 13, 1, 14, 2, 7]
MASK_WORDS = ("mask", "label", "annot", "gt", "segment", "ground")

POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def popcount64(x):
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    return POP8[x.view(np.uint8).reshape(x.shape + (8,))].sum(-1)

class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

def voc_colormap(n=256):
    cmap = np.zeros((n, 3), np.uint8)
    for i in range(n):
        r = g = b = 0; c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j); g |= ((c >> 1) & 1) << (7 - j); b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = (r, g, b)
    return cmap
VOC_CODES = {int(r) << 16 | int(g) << 8 | int(b): i for i, (r, g, b) in enumerate(voc_colormap(16))}

def color_codes(rgb):
    return (rgb[..., 0].astype(np.int64) << 16) | (rgb[..., 1].astype(np.int64) << 8) | rgb[..., 2].astype(np.int64)

def read_mask(path):
    """Returns ('index', array of class ids) or ('color', RGB array) or ('bad', None)."""
    m = Image.open(path)
    if m.mode == "P":
        return "index", np.array(m).astype(np.int64)
    if m.mode in ("L", "I", "I;16", "I;16B", "F", "1"):
        arr = np.array(m).astype(np.int64)
    else:
        rgb = np.array(m.convert("RGB"))
        if np.array_equal(rgb[..., 0], rgb[..., 1]) and np.array_equal(rgb[..., 1], rgb[..., 2]):
            arr = rgb[..., 0].astype(np.int64)
        else:
            return "color", rgb
    if arr.max() > 15:   # masks saved scaled up for viewing (e.g. id*17); the scale is found across all masks
        return "scaled", arr
    return "index", arr

def norm_key(stem):
    s = stem.lower()
    for w in ("mask", "image", "img", "label", "annotation", "annot", "groundtruth", "ground_truth", "segmentation", "_gt", "gt_"):
        s = s.replace(w, "")
    return re.sub(r"[^a-z0-9]", "", s)

def split_hint(rel_parts):
    for p in rel_parts:
        q = p.lower()
        for h in ("train", "test", "val"):
            if h in q:
                return h
    return ""

def phash_int(img):
    return int(str(imagehash.phash(img)), 16)

def dihedral_hashes(img):
    hs = []
    for k in range(4):
        r = img.rotate(90 * k, expand=True)
        hs += [phash_int(r), phash_int(ImageOps.mirror(r))]
    return hs

def build_dwsd_manifest(root):
    files = list_images(root)
    masks, images = {}, {}
    for p in files:
        rel = Path(os.path.relpath(p, root))
        is_mask = any(w in part.lower() for part in rel.parts for w in MASK_WORDS)
        key = (split_hint(rel.parts[:-1]), norm_key(rel.stem))
        (masks if is_mask else images)[key] = p
    pairs = [(images[k], masks[k]) for k in images if k in masks]
    STATS["dwsd_files"] = dict(images=len(images), masks=len(masks), paired=len(pairs))
    print(f"DWSD: {len(images)} images, {len(masks)} masks, {len(pairs)} paired")
    if not pairs:
        print("Could not pair DWSD images with masks. Folder sample:", [os.path.relpath(f, root) for f in files[:15]])
        return None

    # 1) decode all masks to class-id arrays
    decoded, color_masks, scaled, kinds = {}, {}, {}, []
    for img_p, m_p in pairs:
        kind, arr = read_mask(m_p)
        kinds.append(kind)
        if kind == "index": decoded[img_p] = arr
        elif kind == "color": color_masks[img_p] = arr
        elif kind == "scaled": scaled[img_p] = arr
    if scaled:
        vals = np.unique(np.concatenate([np.unique(a) for a in scaled.values()]))
        nz = vals[vals > 0]
        g = int(np.gcd.reduce(nz)) if len(nz) else 1
        if g > 1 and vals.max() // g <= 15:
            for img_p, a in scaled.items():
                decoded[img_p] = a // g
            STATS["dwsd_mask_mapping"] = f"greyscale scaled by {g}"
        else:
            print(f"⚠ {len(scaled)} DWSD masks have values above 15 with no common scale; they are skipped.")
    if color_masks:
        known = total = 0
        for rgb in color_masks.values():
            codes = color_codes(rgb); nz = codes != 0
            total += int(nz.sum()); known += int(np.isin(codes[nz], list(VOC_CODES)).sum())
        if total and known / total >= 0.9:
            mapping, how = VOC_CODES, "voc_palette"
        else:   # unknown palette: match colours to classes by pixel-count rank (flagged in stats)
            counts = {}
            for rgb in color_masks.values():
                u, c = np.unique(color_codes(rgb), return_counts=True)
                for a, b in zip(u, c):
                    if a != 0: counts[int(a)] = counts.get(int(a), 0) + int(b)
            top = sorted(counts, key=counts.get, reverse=True)[:len(DWSD_PIXEL_RANK)]
            mapping, how = {code: cid for code, cid in zip(top, DWSD_PIXEL_RANK)}, "inferred_by_pixel_rank"
            print("⚠ DWSD colour palette unknown; classes inferred by pixel-count rank. Check the crops grid below.")
        for img_p, rgb in color_masks.items():
            codes = color_codes(rgb)
            arr = np.zeros(codes.shape, np.int64)
            for code, cid in mapping.items():
                arr[codes == code] = cid
            decoded[img_p] = arr
        STATS["dwsd_mask_mapping"] = how
    STATS["dwsd_mask_kinds"] = {k: kinds.count(k) for k in set(kinds)}
    print("Mask formats:", STATS["dwsd_mask_kinds"])

    # 2) group parent photos that are flips/rotations/near-copies of each other
    parents = sorted(decoded)
    h8, h0 = [], []
    for p in parents:
        with Image.open(p) as im:
            hs = dihedral_hashes(im.convert("RGB"))
        h8.append(hs); h0.append(hs[0])
    H8, H0 = np.array(h8, dtype=np.uint64), np.array(h0, dtype=np.uint64)
    uf = UnionFind(len(parents))
    for i in range(len(parents)):
        d = popcount64(H8[i][:, None] ^ H0[None, :]).min(0)
        for j in np.nonzero(d <= DWSD_GROUP_THRESHOLD)[0]:
            if j != i: uf.union(i, int(j))
    group_of = {p: f"DWSD_{uf.find(i)}" for i, p in enumerate(parents)}
    STATS["dwsd_parent_photos"] = len(parents)
    STATS["dwsd_parent_groups"] = len(set(group_of.values()))

    # 3) one crop per connected object, padded by 10%
    from scipy import ndimage
    rows, skipped = [], {"too_small": 0, "excluded_class": 0}
    for p in parents:
        arr = decoded[p]
        with Image.open(p) as im:
            W, H = im.size
        if arr.shape != (H, W):
            arr = np.array(Image.fromarray(arr.astype(np.uint8)).resize((W, H), Image.NEAREST)).astype(np.int64)
        rel = os.path.relpath(p, root)
        k = 0
        for cid in np.unique(arr):
            name = DWSD_CLASSES.get(int(cid))
            if name is None:
                continue
            ours = DWSD_TO_OURS[name]
            lab, _ = ndimage.label(arr == cid, structure=np.ones((3, 3)))
            for i, sl in enumerate(ndimage.find_objects(lab), start=1):
                if sl is None: continue
                y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
                area = int((lab[sl] == i).sum())
                if min(y1 - y0, x1 - x0) < MIN_CROP_SIDE or area < MIN_CROP_AREA:
                    skipped["too_small"] += 1; continue
                if ours is None:
                    skipped["excluded_class"] += 1; continue
                py, px = int(0.1 * (y1 - y0)), int(0.1 * (x1 - x0))
                bbox = (max(0, x0 - px), max(0, y0 - py), min(W, x1 + px), min(H, y1 + py))
                rows.append(dict(source="DWSD", src_path=p, rel=f"{rel}#{k}", orig_class=name, label=ours,
                                 bbox=bbox, group=group_of[p]))
                k += 1
    df = pd.DataFrame(rows)
    STATS["dwsd_crops_skipped"] = skipped
    if df.empty:
        print("No usable DWSD crops found.")
        return None
    df["id"] = [stable_id("DWSD", r) for r in df.rel]
    if QUICK_TEST:
        df = df.sample(frac=1, random_state=SEED).groupby("label").head(12)
    STATS["dwsd_crops_by_class"] = {k: int(v) for k, v in df.groupby("orig_class").size().items()}
    print(f"DWSD crops: {len(df)} from {STATS['dwsd_parent_photos']} photos in {STATS['dwsd_parent_groups']} groups;",
          "skipped:", skipped)
    return df.reset_index(drop=True)

if not DATA_READY:
    df_dwsd = None
    if SOURCES.get("dwsd"):
        try:
            df_dwsd = build_dwsd_manifest(SOURCES["dwsd"])
        except Exception as e:
            print(f"✗ DWSD processing failed ({type(e).__name__}: {e}). Continuing without the Indian set.")
    if df_dwsd is not None:
        display(df_dwsd.groupby(["orig_class", "label"]).size().rename("crops").to_frame())
    save_stats()

In [ ]:
#@title 7 · Clean and standardise every image (RGB JPEG, max side 320 px) and compute perceptual hashes
NORM_MAX_SIDE = 320

def normalize_one(job):
    src, dst, bbox, min_side, img_wh = job
    try:
        with Image.open(src) as im0:
            if bbox is not None:
                im0.load()
                # Annotation boxes refer to either the stored pixels or the EXIF-rotated view; use whichever matches.
                if img_wh and tuple(im0.size) != tuple(img_wh):
                    rot = ImageOps.exif_transpose(im0)
                    im = rot if tuple(rot.size) == tuple(img_wh) else im0
                else:
                    im = im0
                im = im.crop(tuple(int(v) for v in bbox))
            else:
                im0.draft("RGB", (2 * NORM_MAX_SIDE, 2 * NORM_MAX_SIDE))
                im = ImageOps.exif_transpose(im0)
            if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
                im = im.convert("RGBA")
                bg = Image.new("RGB", im.size, (255, 255, 255))
                bg.paste(im, mask=im.getchannel("A"))
                im = bg
            else:
                im = im.convert("RGB")
            w, h = im.size
            if min(w, h) < min_side:
                return None, "too_small"
            im.thumbnail((NORM_MAX_SIDE, NORM_MAX_SIDE), Image.Resampling.LANCZOS)
            im.save(dst, "JPEG", quality=93)
            return (str(imagehash.phash(im)), w, h), None
    except Exception as e:
        return None, f"unreadable ({type(e).__name__})"

def normalize_all(df):
    jobs, out_rel = [], []
    for r in df.itertuples():
        rel = f"{r.source}/{r.id}.jpg"
        os.makedirs(f"{NORM_DIR}/{r.source}", exist_ok=True)
        out_rel.append(rel)
        bbox = r.bbox if isinstance(r.bbox, tuple) else None
        wh = getattr(r, "img_wh", None)
        wh = wh if isinstance(wh, tuple) else None
        jobs.append((r.src_path, f"{NORM_DIR}/{rel}", bbox, MIN_CROP_SIDE if bbox else 24, wh))
    results, t0 = [], time.time()
    with ThreadPoolExecutor(max_workers=min(16, 4 * (os.cpu_count() or 2))) as ex:
        for i, res in enumerate(ex.map(normalize_one, jobs, chunksize=32)):
            results.append(res)
            if (i + 1) % 5000 == 0:
                print(f"  {i + 1}/{len(jobs)} images ({time.time() - t0:.0f}s)")
    df = df.copy()
    df["norm_rel"] = out_rel
    df["phash"] = [r[0][0] if r[0] else None for r in results]
    df["src_w"] = [r[0][1] if r[0] else None for r in results]
    df["src_h"] = [r[0][2] if r[0] else None for r in results]
    fails = pd.Series([r[1] for r in results if r[1]]).value_counts().to_dict()
    STATS["cleaning_dropped"] = {k: int(v) for k, v in fails.items()}
    print(f"Cleaned {len(df)} images in {time.time() - t0:.0f}s; dropped: {fails}")
    return df[df.phash.notna()].reset_index(drop=True)

if not DATA_READY:
    parts = [df_web] + ([df_dwsd.assign(img_wh=None)] if df_dwsd is not None else [])
    df_all = normalize_all(pd.concat(parts, ignore_index=True))

In [ ]:
#@title 8 · Remove duplicates across all sources, balance classes across sources, make frozen grouped splits
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

def near_duplicate_pairs(hashes, thr, chunk=512):
    H = np.array(hashes, dtype=np.uint64)
    out = []
    for s in range(0, len(H), chunk):
        d = popcount64(H[s:s + chunk, None] ^ H[None, :])
        ii, jj = np.nonzero(d <= thr)
        ii = ii + s
        keep = jj > ii
        out.append(np.stack([ii[keep], jj[keep], d[ii[keep] - s, jj[keep]]], 1))
    return np.concatenate(out) if out else np.zeros((0, 3), np.int64)

ALL_CSV = f"{DATA_DIR}/all_images_before_dedup.csv"
PAIRS_CSV = f"{RESULTS_DIR}/duplicate_pairs.csv.gz"   # every pair up to distance 10
AUDIT_DIR = f"{RESULTS_DIR}/dedup_audit"

def dedup(df):
    """Near-duplicates anywhere (same photo in two datasets, video frames, re-uploads) collapse to one copy.
    Greedy and non-transitive: images are visited real-photo sources first; an image is dropped if an image already
    kept is within the threshold. If the two carry different labels, both are dropped (the label is ambiguous).
    Every pair up to distance max(DUP_SWEEP) is saved for the duplicate audit."""
    df = df.reset_index(drop=True)
    hashes = [int(h, 16) for h in df.phash]
    pairs = near_duplicate_pairs(hashes, max(DUP_SWEEP))
    src, lab, ids = df.source.values, df.label.values, df.id.values
    pd.DataFrame(dict(id_a=ids[pairs[:, 0]], id_b=ids[pairs[:, 1]], source_a=src[pairs[:, 0]], source_b=src[pairs[:, 1]],
                      label_a=lab[pairs[:, 0]], label_b=lab[pairs[:, 1]], distance=pairs[:, 2],
                      rel_a=df.norm_rel.values[pairs[:, 0]], rel_b=df.norm_rel.values[pairs[:, 1]])
                 ).to_csv(PAIRS_CSV, index=False)
    sweep = []
    for t in DUP_SWEEP:
        p = pairs[pairs[:, 2] <= t]
        sweep.append(dict(threshold=t, pairs=len(p), cross_source=int((src[p[:, 0]] != src[p[:, 1]]).sum()),
                          label_conflicts=int((lab[p[:, 0]] != lab[p[:, 1]]).sum()),
                          images_involved=int(len(np.unique(p[:, :2])))))
    pd.DataFrame(sweep).to_csv(f"{RESULTS_DIR}/dedup_threshold_sweep.csv", index=False)
    display(pd.DataFrame(sweep))
    pairs = pairs[pairs[:, 2] <= PHASH_DUP_THRESHOLD]
    nbrs = [[] for _ in range(len(df))]
    for i, j, _ in pairs:
        nbrs[int(i)].append(int(j)); nbrs[int(j)].append(int(i))
    order = sorted(range(len(df)), key=lambda i: (0 if src[i] in REAL_SOURCES else 1, src[i] in SYNTH_SOURCES, i))
    kept, dropped_conflict = np.zeros(len(df), bool), np.zeros(len(df), bool)
    for i in order:
        hits = [j for j in nbrs[i] if kept[j]]
        if not hits:
            kept[i] = True
        elif any(lab[j] != lab[i] for j in hits):
            dropped_conflict[i] = True
            for j in hits:
                if lab[j] != lab[i]:
                    kept[j] = False; dropped_conflict[j] = True
    by_src = df.assign(kept=kept).groupby("source").kept.agg(["size", "sum"])
    STATS["dedup"] = dict(threshold=PHASH_DUP_THRESHOLD, images_before=len(df), images_after=int(kept.sum()),
                          duplicate_pairs=len(pairs), exact_pairs=int((pairs[:, 2] == 0).sum()) if len(pairs) else 0,
                          cross_source_pairs=int((src[pairs[:, 0]] != src[pairs[:, 1]]).sum()) if len(pairs) else 0,
                          dropped_label_conflict=int(dropped_conflict.sum()),
                          kept_by_source={k: int(v) for k, v in by_src["sum"].items()},
                          before_by_source={k: int(v) for k, v in by_src["size"].items()})
    print("Deduplication:", {k: v for k, v in STATS["dedup"].items() if not k.endswith("by_source")})
    display(by_src.rename(columns={"size": "before", "sum": "after"}))
    return df[kept].reset_index(drop=True)

def dedup_audit_sheets(pairs_csv, n=AUDIT_PAIRS, per_sheet=20):
    """Random cross-source pairs at the dedup threshold, drawn side by side for a manual check."""
    os.makedirs(AUDIT_DIR, exist_ok=True)
    p = pd.read_csv(pairs_csv)
    p = p[(p.source_a != p.source_b) & (p.distance <= PHASH_DUP_THRESHOLD)]
    p = p.sample(min(n, len(p)), random_state=SEED).reset_index(drop=True)
    p.index.name = "pair_no"
    p.to_csv(f"{AUDIT_DIR}/audit_pairs.csv")
    for s0 in range(0, len(p), per_sheet):
        chunk = p.iloc[s0:s0 + per_sheet]
        fig, axes = plt.subplots(len(chunk) // 4 + (len(chunk) % 4 > 0), 8, figsize=(16, 2.3 * (len(chunk) // 4 + 1)))
        axes = np.atleast_2d(axes)
        for a in axes.flat: a.axis("off")
        for k, (no, r) in enumerate(chunk.iterrows()):
            for side, (rel, s_, l_) in enumerate([(r.rel_a, r.source_a, r.label_a), (r.rel_b, r.source_b, r.label_b)]):
                a = axes[k // 4, (k % 4) * 2 + side]
                try:
                    a.imshow(Image.open(f"{NORM_DIR}/{rel}"))
                except Exception:
                    pass
                a.set_title(f"#{no}{'ab'[side]} {s_}:{l_} d={r.distance}", fontsize=6.5)
        plt.tight_layout(); plt.savefig(f"{AUDIT_DIR}/sheet_{s0 // per_sheet:02d}.jpg", dpi=80); plt.close(fig)
    print(f"✓ {len(p)} cross-source pairs drawn to {AUDIT_DIR}")

def balanced_cap(df, cap):
    """Per class, take photo groups round-robin across sources so no single dataset dominates a class."""
    out = []
    rng = np.random.default_rng(SEED)
    for lab, g in df.groupby("label"):
        if len(g) <= cap:
            out.append(g); continue
        queues = {}
        for src, gs in g.groupby("source"):
            ind = gs.groupby("group").indices
            grp = list(ind); rng.shuffle(grp)
            queues[src] = [gs.index[ind[x]].tolist() for x in grp]
        chosen, n = [], 0
        while n < cap and any(queues.values()):
            for src in list(queues):
                if queues[src] and n < cap:
                    idx = queues[src].pop(0); chosen += idx; n += len(idx)
        out.append(g.loc[chosen])
    return pd.concat(out)

def grouped_split(df, test_frac, val_frac):
    """Stratified by class|source, grouped by parent photo: crops of one photo never cross splits."""
    n_folds = 10
    strat = (df.label + "|" + df.source).values
    folds = np.full(len(df), -1)
    try:
        sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
        for f, (_, idx) in enumerate(sgkf.split(df, strat, df.group.values)):
            folds[idx] = f
    except ValueError:
        rng = np.random.default_rng(SEED)
        gid = {g: rng.integers(n_folds) for g in df.group.unique()}
        folds = df.group.map(gid).values
    n_test, n_val = round(test_frac * n_folds), round(val_frac * n_folds)
    return np.where(folds < n_test, "test", np.where(folds < n_test + n_val, "val", "train"))

def make_splits(df_all):
    main = dedup(df_all[df_all.source != "DWSD"])
    main = balanced_cap(main, CAP_PER_CLASS)
    main["split"] = grouped_split(main, TEST_FRACTION, VAL_FRACTION)
    main.loc[main.source.isin(SYNTH_SOURCES), "split"] = "train"   # synthetic images are never evaluated on
    frames = [main]
    dw = df_all[df_all.source == "DWSD"].copy()
    if len(dw):
        n_groups = dw.group.nunique()
        if n_groups >= 10:
            folds = np.zeros(len(dw), int)
            sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
            for f, (_, idx) in enumerate(sgkf.split(dw, dw.label, dw.group)):
                folds[idx] = f
            test_mask = folds < round(10 * DWSD_TEST_FRACTION)
        else:
            gss = GroupShuffleSplit(n_splits=1, test_size=DWSD_TEST_FRACTION, random_state=SEED)
            _, idx = next(gss.split(dw, dw.label, dw.group))
            test_mask = np.zeros(len(dw), bool); test_mask[idx] = True
        dw["split"] = np.where(test_mask, "dwsd_test", "train")
        frames.append(dw)
    out = pd.concat(frames, ignore_index=True)
    out["y"] = out.label.map(CLASS_TO_IDX).astype(int)
    out["real"] = out.source.isin(REAL_SOURCES)
    cols = ["id", "source", "orig_class", "label", "y", "split", "group", "real", "norm_rel", "phash", "src_w", "src_h"]
    return out[cols]

def leakage_audit(df):
    """Near-duplicates (pHash distance <= threshold) between train and each held-out split."""
    res = {}
    tr = df[df.split == "train"]
    H_tr = np.array([int(h, 16) for h in tr.phash], dtype=np.uint64)
    for s in ["val", "test", "dwsd_test"]:
        ho = df[df.split == s]
        if not len(ho):
            continue
        H = np.array([int(h, 16) for h in ho.phash], dtype=np.uint64)
        n = 0
        for a in range(0, len(H), 256):
            n += int((popcount64(H[a:a + 256, None] ^ H_tr[None, :]).min(1) <= PHASH_DUP_THRESHOLD).sum())
        res[s] = n
    STATS["leakage_audit_near_duplicates_vs_train"] = res
    print("Held-out images with a near-duplicate in train:", res)

if not DATA_READY:
    df_all[["id", "source", "orig_class", "label", "group", "norm_rel", "phash"]].to_csv(ALL_CSV, index=False)
    df_split = make_splits(df_all)
    leakage_audit(df_split)
    dedup_audit_sheets(PAIRS_CSV)
    df_split.to_csv(SPLITS_CSV, index=False)
    print("Saving prepared images to Drive (one-time; includes removed duplicates for the leakage experiment) ...")
    with tarfile.open(NORM_TAR + ".part", "w") as t:
        for rel in df_all.norm_rel:
            t.add(f"{NORM_DIR}/{rel}", arcname=rel)
    os.replace(NORM_TAR + ".part", NORM_TAR)
    save_stats()
    print(f"✓ Frozen splits saved to {SPLITS_CSV}")
    display(df_split.pivot_table(index="label", columns="split", values="id", aggfunc="count", fill_value=0))

In [ ]:
#@title 9 · Data summary (tables and figures for the report)
df_split = pd.read_csv(SPLITS_CSV)
df_split["path"] = NORM_DIR + "/" + df_split.norm_rel
missing = (~df_split.path.map(os.path.exists)).sum()
assert missing == 0, f"{missing} prepared images are missing; delete {NORM_TAR} and {SPLITS_CSV} and rerun."

table = df_split.pivot_table(index="label", columns=["split", "source"], values="id", aggfunc="count", fill_value=0)
table = table.reindex(CLASSES)
table.to_csv(f"{RESULTS_DIR}/data_counts.csv")
display(table)
STATS["data_counts"] = {s: {k: int(v) for k, v in g.groupby("label").size().items()}
                        for s, g in df_split.groupby("split")}
STATS["data_totals"] = {s: int(n) for s, n in df_split.groupby("split").size().items()}
STATS["data_by_source"] = {s: int(n) for s, n in df_split.groupby("source").size().items()}
save_stats()

fig, ax = plt.subplots(figsize=(11, 4))
cnt = df_split.groupby(["label", "source"]).size().unstack(fill_value=0).reindex(CLASSES)
cnt.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_xlabel(""); ax.set_ylabel("Images"); ax.set_title("Images per class and source (after cleaning, deduplication and capping)")
ax.set_xticklabels([CLASS_DISPLAY[c] for c in CLASSES], rotation=30, ha="right")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/data_distribution.png", dpi=200); plt.show()

def sample_grid(frame, title, path, per_class=3):
    cls = [c for c in CLASSES if (frame.label == c).any()]
    fig, axes = plt.subplots(len(cls), per_class, figsize=(per_class * 2.1, len(cls) * 2.1))
    axes = np.atleast_2d(axes)
    for r, c in enumerate(cls):
        g = frame[frame.label == c]
        picks = g.assign(k=g.groupby("source").cumcount()).sort_values("k", kind="stable").head(per_class)  # mix sources
        for k in range(per_class):
            a = axes[r, k]; a.axis("off")
            if k < len(picks):
                row = picks.iloc[k]
                a.imshow(Image.open(row.path)); a.set_title(f"{c} · {row.source}", fontsize=7)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

sample_grid(df_split.sample(frac=1, random_state=1), "Example training images", f"{FIG_DIR}/data_samples.png")

# Equitable distribution: each class's share in train, validation and test (should be near-identical)
share = df_split[df_split.split.isin(["train", "val", "test"])].groupby(["split", "label"]).size().unstack(fill_value=0)
share = share.div(share.sum(1), axis=0).reindex(columns=CLASSES).loc[["train", "val", "test"]]
share.to_csv(f"{RESULTS_DIR}/class_share_by_split.csv")
fig, ax = plt.subplots(figsize=(11, 3.6))
share.T.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_ylabel("Share of split"); ax.set_xlabel(""); ax.set_title("Class share in train / validation / test (stratified, grouped split)")
ax.set_xticklabels([CLASS_DISPLAY[c] for c in CLASSES], rotation=30, ha="right")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/class_share_by_split.png", dpi=200); plt.show()
STATS["real_photo_share"] = {s: float(g.real.mean()) for s, g in df_split.groupby("split")}
print("Share of real photos per split:", STATS["real_photo_share"])
if (df_split.source == "DWSD").any():
    sample_grid(df_split[df_split.source == "DWSD"].sample(frac=1, random_state=1),
                "DWSD (Kolkata) object crops", f"{FIG_DIR}/data_samples_dwsd.png", per_class=4)

## Step 2 · Train the models (EfficientNetV2-B0 with three seeds, MobileNetV2, a CNN from scratch)
Each finished model is saved to Drive immediately. If the session disconnects, run all cells again and
training continues with the next unfinished model.

In [ ]:
#@title 10 · TensorFlow and input pipeline
import tensorflow as tf
import keras
from keras import layers

GPUS = tf.config.list_physical_devices("GPU")
for g in GPUS:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
STATS["environment"] = dict(tensorflow=tf.__version__, keras=keras.__version__, python=sys.version.split()[0],
                            gpu=[tf.config.experimental.get_device_details(g).get("device_name", "GPU") for g in GPUS] or "none")
print(STATS["environment"])
if not GPUS:
    print("⚠ No GPU found. Use Runtime → Change runtime type → T4 GPU, otherwise training will be very slow.")

for f in glob.glob(f"{CACHE_DIR}/*.lockfile"):
    os.remove(f)

SPLITS = {s: df_split[df_split.split == s].reset_index(drop=True) for s in ["train", "val", "test", "dwsd_test"]}
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path):
    """The single preprocessing used everywhere (training, evaluation, Grad-CAM and the app)."""
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize_with_pad(img, IMG_SIZE, IMG_SIZE, antialias=True)
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)

def make_ds(frame, name, training=False):
    paths = frame.path.values.astype(str)
    ys = frame.y.values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE)
    key = hashlib.md5("".join(frame.id).encode()).hexdigest()[:10]
    ds = ds.cache(f"{CACHE_DIR}/{name}_{key}")
    if training:
        ds = ds.shuffle(min(len(frame), 2048), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

def make_balanced_ds(frame, name):
    """Equal share of every class in each batch: each class is its own cached, shuffled, repeating stream.
    Rare classes are therefore seen more often, each time with fresh random augmentation (inside the model)."""
    streams, weights = [], []
    for c in sorted(frame.y.unique()):
        f = frame[frame.y == c]
        ds = tf.data.Dataset.from_tensor_slices((f.path.values.astype(str), f.y.values.astype("int32")))
        ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE)
        key = hashlib.md5("".join(f.id).encode()).hexdigest()[:10]
        ds = ds.cache(f"{CACHE_DIR}/{name}_c{c}_{key}").shuffle(min(len(f), 1024), seed=SEED).repeat()
        streams.append(ds); weights.append(1.0)
    ds = tf.data.Dataset.sample_from_datasets(streams, weights=[w / sum(weights) for w in weights], seed=SEED)
    ds = ds.batch(BATCH_SIZE).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

DS = {s: make_ds(f, s, training=False) for s, f in SPLITS.items() if len(f) and s != "train"}
if BALANCED_BATCHES:
    DS["train"] = make_balanced_ds(SPLITS["train"], "train")
    STEPS_PER_EPOCH = math.ceil(len(SPLITS["train"]) / BATCH_SIZE)
    CLASS_WEIGHTS = None
else:
    DS["train"] = make_ds(SPLITS["train"], "train", training=True)
    STEPS_PER_EPOCH = None
    counts = np.bincount(SPLITS["train"].y, minlength=NUM_CLASSES)
    CLASS_WEIGHTS = {i: float(np.clip(counts.sum() / (NUM_CLASSES * c), 0.5, 5.0)) if c else 1.0
                     for i, c in enumerate(counts)}
print({s: len(f) for s, f in SPLITS.items()})
print("Balanced batches:", BALANCED_BATCHES, "| steps per epoch:", STEPS_PER_EPOCH)

In [ ]:
#@title 11 · Model definitions
@keras.saving.register_keras_serializable(package="WasteVision")
class ColorJitter(layers.Layer):
    """Per-image random brightness, contrast, saturation and hue (training only).
    Mimics glare, dim rooms, cheap phone cameras and food stains."""
    def __init__(self, brightness=0.25, contrast=0.25, saturation=0.35, hue=0.03, **kwargs):
        super().__init__(**kwargs)
        self.brightness, self.contrast, self.saturation, self.hue = brightness, contrast, saturation, hue

    def call(self, x, training=None):
        if not training:
            return x
        b = tf.shape(x)[0]
        x = tf.clip_by_value(x / 255.0, 0.0, 1.0)
        h, s, v = tf.unstack(tf.image.rgb_to_hsv(x), axis=-1)
        h = tf.math.floormod(h + tf.random.uniform([b, 1, 1], -self.hue, self.hue), 1.0)
        s = tf.clip_by_value(s * tf.random.uniform([b, 1, 1], 1 - self.saturation, 1 + self.saturation), 0.0, 1.0)
        v = tf.clip_by_value(v * tf.random.uniform([b, 1, 1], 1 - self.brightness, 1 + self.brightness), 0.0, 1.0)
        x = tf.image.hsv_to_rgb(tf.stack([h, s, v], axis=-1))
        mean = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
        x = (x - mean) * tf.random.uniform([b, 1, 1, 1], 1 - self.contrast, 1 + self.contrast) + mean
        return tf.clip_by_value(x, 0.0, 1.0) * 255.0

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        return {**super().get_config(), "brightness": self.brightness, "contrast": self.contrast,
                "saturation": self.saturation, "hue": self.hue}

def make_augmenter():
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical", name="aug_flip"),
        layers.RandomRotation(0.08, fill_mode="reflect", name="aug_rotate"),
        layers.RandomZoom((-0.15, 0.15), fill_mode="reflect", name="aug_zoom"),
        layers.RandomTranslation(0.08, 0.08, fill_mode="reflect", name="aug_translate"),
        ColorJitter(name="aug_color"),
    ], name="augment")

def scratch_backbone():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="scratch_in")
    x = inp
    for b, f in enumerate([32, 64, 128, 192, 256]):
        for k in range(2):
            x = layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"b{b}_conv{k}")(x)
            x = layers.BatchNormalization(name=f"b{b}_bn{k}")(x)
            x = layers.ReLU(name=f"b{b}_relu{k}")(x)
        x = layers.MaxPooling2D(name=f"b{b}_pool")(x)
    return keras.Model(inp, x, name="backbone")

def make_head(feature_shape):
    inp = keras.Input(feature_shape, name="features")
    x = layers.GlobalAveragePooling2D(name="head_gap")(inp)
    x = layers.Dropout(0.2, name="head_drop1")(x)
    x = layers.Dense(128, activation="relu", name="head_dense")(x)
    x = layers.Dropout(0.3, name="head_drop2")(x)
    out = layers.Dense(NUM_CLASSES, name="logits")(x)   # logits; softmax (with temperature) applied at inference
    return keras.Model(inp, out, name="head")

def build_model(arch, weights=None):
    """Returns (training model with augmentation, inference model, backbone).
    Both models share the same preprocessing, backbone and head layers."""
    shape = (IMG_SIZE, IMG_SIZE, 3)
    arch = base_arch(arch)
    if arch == "scratch_cnn":
        preproc, backbone, frozen_bn = layers.Rescaling(1 / 255.0, name="preproc"), scratch_backbone(), False
    elif arch == "mobilenetv2":
        preproc = layers.Rescaling(1 / 127.5, offset=-1.0, name="preproc")
        base = keras.applications.MobileNetV2(input_shape=shape, include_top=False, weights=weights)
        backbone, frozen_bn = keras.Model(base.input, base.output, name="backbone"), True
    elif arch == "efficientnetv2b0":
        preproc = layers.Identity(name="preproc")   # EfficientNetV2 scales 0-255 input internally
        base = keras.applications.EfficientNetV2B0(input_shape=shape, include_top=False, weights=weights,
                                                   include_preprocessing=True)
        backbone, frozen_bn = keras.Model(base.input, base.output, name="backbone"), True
    else:
        raise ValueError(arch)
    head = make_head(backbone.output_shape[1:])

    def forward(x):
        x = preproc(x)
        x = backbone(x, training=False) if frozen_bn else backbone(x)
        return head(x)

    inp = keras.Input(shape, name="image")
    train_model = keras.Model(inp, forward(make_augmenter()(inp)), name=f"{arch}_train")
    inp2 = keras.Input(shape, name="image")
    infer_model = keras.Model(inp2, forward(inp2), name=arch)
    return train_model, infer_model, backbone

def unfreeze_top(backbone, n):
    backbone.trainable = True
    for l in backbone.layers[:-n]:
        l.trainable = False
    for l in backbone.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False   # keep pretrained batch-norm statistics fixed while fine-tuning

for arch in dict.fromkeys(base_arch(a) for a in MODELS):
    _, m, _ = build_model(arch, weights=None)
    print(f"{arch:18s} parameters: {m.count_params() / 1e6:.2f} M")
keras.backend.clear_session()

In [ ]:
#@title 12 · Train (skips models already saved on Drive)
# Seeded runs use a different random seed for the head initialisation and the augmentation draws.
class KeepBest(keras.callbacks.Callback):
    """Restore the weights from the epoch with the lowest validation loss."""
    def __init__(self):
        super().__init__(); self.best, self.weights = np.inf, None
    def on_epoch_end(self, epoch, logs=None):
        v = (logs or {}).get("val_loss")
        if v is not None and v < self.best:
            self.best, self.weights = v, self.model.get_weights()
    def on_train_end(self, logs=None):
        if self.weights is not None:
            self.model.set_weights(self.weights)

def compile_model(m, lr):
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")], jit_compile=False)

def fit(m, train_ds, epochs, initial_epoch, patience, log_path, steps=None, class_weight=None):
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=max(1, patience // 2),
                                             min_lr=1e-7, verbose=1),
           KeepBest(), keras.callbacks.CSVLogger(log_path, append=True)]
    return m.fit(train_ds, validation_data=DS["val"], epochs=initial_epoch + epochs, initial_epoch=initial_epoch,
                 steps_per_epoch=steps, class_weight=class_weight, callbacks=cbs, verbose=1 if IN_COLAB else 2)

def to_floats(h):
    return {k: [float(v) for v in vals] for k, vals in h.history.items()}

for arch in MODELS:
    out = f"{MODELS_DIR}/{arch}"
    os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/infer.keras") and not RETRAIN:
        print(f"✓ {arch}: already trained, skipping"); continue
    print(f"\n========== Training {arch} ==========")
    keras.utils.set_random_seed(seed_of(arch))
    scratch = base_arch(arch) == "scratch_cnn"
    train_m, infer_m, backbone = build_model(arch, weights=None if scratch else PRETRAINED_WEIGHTS)
    log = f"{out}/training_log.csv"
    if os.path.exists(log): os.remove(log)
    t0, phases = time.time(), {}
    if scratch:
        compile_model(train_m, LR_SCRATCH)
        phases["full"] = to_floats(fit(train_m, DS["train"], EPOCHS_SCRATCH, 0, 4, log, STEPS_PER_EPOCH, CLASS_WEIGHTS))
    else:
        backbone.trainable = False
        compile_model(train_m, LR_HEAD)
        phases["head"] = to_floats(fit(train_m, DS["train"], EPOCHS_HEAD, 0, 3, log, STEPS_PER_EPOCH, CLASS_WEIGHTS))
        unfreeze_top(backbone, UNFREEZE_LAST_LAYERS)
        compile_model(train_m, LR_FINETUNE)
        phases["finetune"] = to_floats(fit(train_m, DS["train"], EPOCHS_FINETUNE,
                                           len(phases["head"]["loss"]), 4, log, STEPS_PER_EPOCH, CLASS_WEIGHTS))
    infer_m.save(f"{out}/infer.keras")
    info = dict(arch=arch, train_seconds=time.time() - t0, params=int(infer_m.count_params()),
                trainable_params_final=int(sum(np.prod(v.shape) for v in train_m.trainable_weights)),
                phases=phases, seed=seed_of(arch), pretrained=None if scratch else PRETRAINED_WEIGHTS)
    json.dump(info, open(f"{out}/train_info.json", "w"), indent=1)
    print(f"✓ {arch} trained in {info['train_seconds'] / 60:.1f} min and saved to {out}")
    del train_m, infer_m, backbone
    keras.backend.clear_session()

## Step 3 · Evaluate: calibration, streams, safety rule, uncertainty, Indian test set

In [ ]:
#@title 13 · Evaluation helpers
from scipy.special import softmax, log_softmax
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def fit_temperature(logits, y):
    def nll(log_t):
        return -log_softmax(logits / np.exp(log_t), axis=1)[np.arange(len(y)), y].mean()
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))

def ece(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def stream_probs(p):
    return p @ STREAM_MATRIX

def risk_coverage(conf, correct):
    order = np.argsort(-conf)
    c = correct[order].astype(float)
    k = np.arange(1, len(c) + 1)
    return k / len(c), np.cumsum(c) / k, conf[order]

def pick_threshold(conf, correct, target):
    """Lowest confidence threshold whose answered items still reach the target accuracy (max coverage)."""
    cov, acc, th = risk_coverage(conf, correct)
    ok = np.nonzero(acc >= target)[0]
    if not len(ok):
        return float(th[max(0, int(0.8 * len(th)) - 1)]), False
    return float(th[ok.max()]), True

def selective(conf, correct, t):
    m = conf >= t
    return dict(coverage=float(m.mean()), accuracy_answered=float(correct[m].mean()) if m.any() else float("nan"))

def metrics(p, y):
    pred = p.argmax(1)
    sp = stream_probs(p)
    s_true = CLASS_STREAM_IDX[y]
    s_pred = sp.argmax(1)
    is_sc = s_true == SC
    out = dict(n=int(len(y)), class_acc=float(accuracy_score(y, pred)),
               macro_f1=float(f1_score(y, pred, average="macro", labels=np.unique(y), zero_division=0)),
               stream_acc=float((s_pred == s_true).mean()),
               stream_acc_top_class=float((CLASS_STREAM_IDX[pred] == s_true).mean()))
    for r in SENSITIVITY_COST_RATIOS:
        t = 1 / (1 + r)
        flag = sp[:, SC] >= t
        out[f"sc_recall@{t:.2f}"] = float(flag[is_sc].mean()) if is_sc.any() else float("nan")
        out[f"sc_false_alarm@{t:.2f}"] = float(flag[~is_sc].mean()) if (~is_sc).any() else float("nan")
    out["sc_recall_argmax"] = float((s_pred[is_sc] == SC).mean()) if is_sc.any() else float("nan")
    return out

def group_bootstrap_ci(frame, fn, rounds=BOOTSTRAP_ROUNDS):
    """95% interval resampling whole photo groups (crops from one photo stay together)."""
    groups = frame.group.values
    uniq = np.unique(groups)
    idx = {g: np.nonzero(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(rounds):
        ix = np.concatenate([idx[g] for g in rng.choice(uniq, len(uniq), replace=True)])
        vals.append(fn(ix))
    return [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))]

def measure_latency(model, runs=30):
    x = tf.random.uniform((1, IMG_SIZE, IMG_SIZE, 3), 0, 255)
    res = {}
    for name, dev in ([("gpu", "/GPU:0")] if GPUS else []) + [("cpu", "/CPU:0")]:
        with tf.device(dev):
            f = tf.function(lambda t: model(t, training=False))
            for _ in range(5): f(x).numpy()
            t0 = time.perf_counter()
            for _ in range(runs): f(x).numpy()
            res[f"latency_ms_{name}"] = (time.perf_counter() - t0) / runs * 1000
    return res

In [ ]:
#@title 14 · Evaluate all models and pick the best one (chosen on validation macro-F1, never on test)
RESULTS, PRED = {}, {}
for arch in MODELS:
    path = f"{MODELS_DIR}/{arch}/infer.keras"
    if not os.path.exists(path):
        print(f"⚠ {arch} not trained yet, skipping"); continue
    model = keras.models.load_model(path, compile=False)
    logits = {s: model.predict(DS[s], verbose=0) for s in DS if s != "train"}
    y = {s: SPLITS[s].y.values for s in logits}
    T = fit_temperature(logits["val"], y["val"])
    raw = {s: softmax(l, 1) for s, l in logits.items()}
    cal = {s: softmax(l / T, 1) for s, l in logits.items()}
    r = dict(temperature=T)
    for s in logits:
        r[s] = metrics(cal[s], y[s])
    r["ece_val_before"], r["ece_val_after"] = ece(raw["val"], y["val"]), ece(cal["val"], y["val"])
    r["ece_test_before"], r["ece_test_after"] = ece(raw["test"], y["test"]), ece(cal["test"], y["test"])

    # "Not sure" thresholds chosen on validation, then checked on test
    sv, st = stream_probs(cal["val"]), stream_probs(cal["test"])
    t_stream, ok_s = pick_threshold(sv.max(1), sv.argmax(1) == CLASS_STREAM_IDX[y["val"]], TARGET_STREAM_ACC)
    t_class, ok_c = pick_threshold(cal["val"].max(1), cal["val"].argmax(1) == y["val"], TARGET_CLASS_ACC)
    r["not_sure_stream_threshold"], r["not_sure_class_threshold"] = t_stream, t_class
    r["not_sure_targets_met_on_val"] = dict(stream=ok_s, cls=ok_c)
    r["test_selective_stream"] = selective(st.max(1), st.argmax(1) == CLASS_STREAM_IDX[y["test"]], t_stream)
    r["test_selective_class"] = selective(cal["test"].max(1), cal["test"].argmax(1) == y["test"], t_class)

    # Accuracy per data source (shortcut check)
    te = SPLITS["test"]
    for subset, mask in (("test_real", te.real.values), ("test_web", ~te.real.values)):
        if mask.any():
            r[subset] = metrics(cal["test"][mask], y["test"][mask])
    r["test_by_source"] = {src: dict(n=int((te.source == src).sum()),
                                     class_acc=float((cal["test"].argmax(1)[te.source == src] == y["test"][te.source == src]).mean()))
                           for src in te.source.unique()}

    # Indian held-out set with grouped bootstrap intervals
    if "dwsd_test" in cal:
        p, yy, fr = cal["dwsd_test"], y["dwsd_test"], SPLITS["dwsd_test"]
        pred, sp = p.argmax(1), stream_probs(p)
        r["dwsd_ci"] = dict(
            class_acc=group_bootstrap_ci(fr, lambda ix: (pred[ix] == yy[ix]).mean()),
            stream_acc=group_bootstrap_ci(fr, lambda ix: (sp[ix].argmax(1) == CLASS_STREAM_IDX[yy[ix]]).mean()),
            sc_false_alarm=group_bootstrap_ci(fr, lambda ix: (sp[ix, SC] >= SC_THRESHOLD).mean()))
        r["dwsd_groups"] = int(fr.group.nunique())

    info = json.load(open(f"{MODELS_DIR}/{arch}/train_info.json"))
    r.update(params_m=info["params"] / 1e6, size_mb=os.path.getsize(path) / 1e6,
             train_minutes=info["train_seconds"] / 60, **measure_latency(model))
    RESULTS[arch], PRED[arch] = r, dict(raw=raw, cal=cal, y=y)
    print(f"✓ {arch}: test class acc {r['test']['class_acc']:.3f} | stream acc {r['test']['stream_acc']:.3f} | "
          f"T={T:.2f}")
    del model
    keras.backend.clear_session()

json.dump(RESULTS, open(f"{RESULTS_DIR}/results_all_models.json", "w"), indent=1)

def fmt_ci(v, ci):
    return f"{v:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]"

t20 = f"{SC_THRESHOLD:.2f}"
rows = []
for arch, r in RESULTS.items():
    row = {"Model": arch,
           "Val macro-F1": round(r["val"]["macro_f1"], 3),
           "Test class acc": round(r["test"]["class_acc"], 3),
           "Test macro-F1": round(r["test"]["macro_f1"], 3),
           "Test stream acc": round(r["test"]["stream_acc"], 3),
           "Stream acc (top-class lookup)": round(r["test"]["stream_acc_top_class"], 3),
           f"Special Care recall @{t20}": round(r["test"][f"sc_recall@{t20}"], 3),
           f"False alarm @{t20}": round(r["test"][f"sc_false_alarm@{t20}"], 3),
           "ECE test before → after": f"{r['ece_test_before']:.3f} → {r['ece_test_after']:.3f}",
           "Temperature": round(r["temperature"], 2)}
    if "test_real" in r:
        row["Test class acc, real photos"] = round(r["test_real"]["class_acc"], 3)
        row["Test macro-F1, real photos"] = round(r["test_real"]["macro_f1"], 3)
    if "test_web" in r:
        row["Test class acc, web/mixed"] = round(r["test_web"]["class_acc"], 3)
    if "dwsd_test" in r:
        row["Indian class acc [95% CI]"] = fmt_ci(r["dwsd_test"]["class_acc"], r["dwsd_ci"]["class_acc"])
        row["Indian stream acc [95% CI]"] = fmt_ci(r["dwsd_test"]["stream_acc"], r["dwsd_ci"]["stream_acc"])
    row.update({"Params (M)": round(r["params_m"], 2), "Size (MB)": round(r["size_mb"], 1),
                "Latency GPU (ms)": round(r.get("latency_ms_gpu", float("nan")), 1),
                "Latency CPU (ms)": round(r["latency_ms_cpu"], 1), "Train (min)": round(r["train_minutes"], 1)})
    rows.append(row)
COMPARISON = pd.DataFrame(rows).set_index("Model")
COMPARISON.to_csv(f"{RESULTS_DIR}/model_comparison.csv")
display(COMPARISON.T)

BEST = max(RESULTS, key=lambda a: RESULTS[a]["val"]["macro_f1"])
STATS["best_model"] = BEST
save_stats()
print(f"\n★ Best model on validation macro-F1: {BEST}")

In [ ]:
#@title 15 · Figures for the report and paper
COLORS = {"scratch_cnn": "#8172B3", "mobilenetv2": "#4C72B0", "efficientnetv2b0": "#55A868",
          f"{MAIN_ARCH}_seed7": "#8FBF8F", f"{MAIN_ARCH}_seed123": "#2E7D32"}

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for arch in RESULTS:
    log = pd.read_csv(f"{MODELS_DIR}/{arch}/training_log.csv")
    axes[0].plot(log.epoch + 1, log.val_acc, label=f"{arch} (val)", color=COLORS.get(arch))
    axes[0].plot(log.epoch + 1, log.acc, "--", alpha=0.5, color=COLORS.get(arch))
    axes[1].plot(log.epoch + 1, log.val_loss, label=arch, color=COLORS.get(arch))
axes[0].set_title("Accuracy (solid = validation, dashed = training)"); axes[0].set_xlabel("Epoch")
axes[1].set_title("Validation loss"); axes[1].set_xlabel("Epoch")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/training_curves.png", dpi=200); plt.show()

# Confusion matrices for the best model (test)
P = PRED[BEST]
yt, pt = P["y"]["test"], P["cal"]["test"]
cm = confusion_matrix(yt, pt.argmax(1), labels=range(NUM_CLASSES), normalize="true")
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8); ax.set_yticklabels(CLASSES, fontsize=8)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j] >= 0.01:
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if cm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{BEST}: class confusion (test, row-normalised)")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/confusion_classes.png", dpi=200); plt.show()

sp = stream_probs(pt)
scm = confusion_matrix(CLASS_STREAM_IDX[yt], sp.argmax(1), labels=range(len(STREAMS)))
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(scm / np.maximum(scm.sum(1, keepdims=True), 1), cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(len(STREAMS))); ax.set_yticks(range(len(STREAMS)))
ax.set_xticklabels(STREAMS); ax.set_yticklabels(STREAMS)
for i in range(len(STREAMS)):
    for j in range(len(STREAMS)):
        ax.text(j, i, str(scm[i, j]), ha="center", va="center")
ax.set_xlabel("Predicted stream"); ax.set_ylabel("True stream"); ax.set_title(f"{BEST}: SWM stream confusion (test)")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/confusion_streams.png", dpi=200); plt.show()

# Reliability diagram before and after temperature scaling (validation)
fig, ax = plt.subplots(figsize=(4.8, 4.5))
for key, label in [("raw", "before"), ("cal", "after")]:
    pr, yy = P[key]["val"], P["y"]["val"]
    conf, corr = pr.max(1), pr.argmax(1) == yy
    edges = np.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 5:
            xs.append(conf[m].mean()); ys.append(corr[m].mean())
    ax.plot(xs, ys, "o-", label=f"{label} (ECE {ece(pr, yy):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("Predicted confidence"); ax.set_ylabel("Actual accuracy")
ax.set_title(f"{BEST}: calibration (validation)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/calibration.png", dpi=200); plt.show()

# Special Care recall and false alarms vs threshold (all models, test)
taus = np.linspace(0.02, 0.9, 89)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for arch in RESULTS:
    pr, yy = PRED[arch]["cal"]["test"], PRED[arch]["y"]["test"]
    psc, is_sc = stream_probs(pr)[:, SC], CLASS_STREAM_IDX[yy] == SC
    axes[0].plot(taus, [(psc[is_sc] >= t).mean() for t in taus], label=arch, color=COLORS.get(arch))
    axes[1].plot(taus, [(psc[~is_sc] >= t).mean() for t in taus], label=arch, color=COLORS.get(arch))
for ax, title in zip(axes, ["Special Care recall", "False alarm rate (non-hazardous items flagged)"]):
    for r_ in SENSITIVITY_COST_RATIOS:
        ax.axvline(1 / (1 + r_), color="grey", ls=":", lw=1)
        ax.text(1 / (1 + r_), 1.01, f"{r_:g}:1", ha="center", fontsize=8, transform=ax.get_xaxis_transform())
    ax.set_xlabel("Warning threshold on P(Special Care)"); ax.set_title(title, pad=14); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/special_care_threshold.png", dpi=200); plt.show()

# Risk–coverage for the stream decision (all models, test)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
for arch in RESULTS:
    pr, yy = PRED[arch]["cal"]["test"], PRED[arch]["y"]["test"]
    s = stream_probs(pr)
    cov, acc, _ = risk_coverage(s.max(1), s.argmax(1) == CLASS_STREAM_IDX[yy])
    ax.plot(cov, 1 - acc, label=arch, color=COLORS.get(arch))
ax.set_xlabel("Coverage (share of items the model answers)"); ax.set_ylabel("Error rate on answered items")
ax.set_title("Risk–coverage: SWM stream decision (test)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/risk_coverage.png", dpi=200); plt.show()

# Accuracy by data source (best model)
r = RESULTS[BEST]
names = list(r["test_by_source"]) + (["DWSD (Indian held-out)"] if "dwsd_test" in r else [])
vals = [v["class_acc"] for v in r["test_by_source"].values()] + ([r["dwsd_test"]["class_acc"]] if "dwsd_test" in r else [])
fig, ax = plt.subplots(figsize=(max(6, 0.7 * len(names)), 4))
bars = ax.bar(names, vals, color=["#4C72B0"] * (len(names) - ("dwsd_test" in r)) + (["#55A868"] if "dwsd_test" in r else []))
ax.tick_params(axis="x", rotation=40)
if "dwsd_test" in r:
    lo, hi = r["dwsd_ci"]["class_acc"]
    ax.errorbar(len(names) - 1, vals[-1], yerr=[[vals[-1] - lo], [hi - vals[-1]]], color="black", capsize=5)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=7)
ax.set_ylim(0, 1.08); ax.set_ylabel("Class accuracy"); ax.set_title(f"{BEST}: accuracy by image source")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/accuracy_by_source.png", dpi=200); plt.show()

report = classification_report(yt, pt.argmax(1), labels=range(NUM_CLASSES), target_names=CLASSES, zero_division=0, digits=3)
open(f"{RESULTS_DIR}/classification_report_{BEST}.txt", "w").write(report)
print(report)

## Step 4 · Grad-CAM: what the model looks at (correct and wrong predictions)

In [ ]:
#@title 16 · Grad-CAM gallery
BEST_MODEL = keras.models.load_model(f"{MODELS_DIR}/{BEST}/infer.keras", compile=False)
T_BEST = RESULTS[BEST]["temperature"]

def grad_cam(model, images, class_idx=None):
    """images: uint8 or float array (N, 224, 224, 3) in 0-255. Returns heatmaps in [0, 1] and logits."""
    pre, bb, hd = model.get_layer("preproc"), model.get_layer("backbone"), model.get_layer("head")
    x = tf.convert_to_tensor(images, tf.float32)
    with tf.GradientTape() as tape:
        fmap = bb(pre(x), training=False)
        tape.watch(fmap)
        logits = hd(fmap, training=False)
        idx = tf.argmax(logits, 1) if class_idx is None else tf.convert_to_tensor(class_idx, tf.int64)
        score = tf.gather(logits, idx, axis=1, batch_dims=1)
    grads = tape.gradient(score, fmap)
    cam = tf.nn.relu(tf.reduce_sum(tf.reduce_mean(grads, axis=(1, 2), keepdims=True) * fmap, axis=-1))
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam.numpy(), logits.numpy()

def overlay(img, cam, alpha=0.45):
    heat = np.array(Image.fromarray(np.uint8(cam * 255)).resize((img.shape[1], img.shape[0]), Image.BILINEAR)) / 255.0
    colored = matplotlib.colormaps["jet"](heat)[..., :3]
    return np.uint8(np.clip((1 - alpha) * img + alpha * colored * 255, 0, 255))

def load_batch(paths):
    return np.stack([load_image(p).numpy() for p in paths])

def gallery(frame, probs, title, path, per_class=2, export_prefix=None):
    pred, conf, yy = probs.argmax(1), probs.max(1), frame.y.values
    picks = []
    for c in range(NUM_CLASSES):
        m = yy == c
        if not m.any(): continue
        ok = np.nonzero(m & (pred == c))[0]; bad = np.nonzero(m & (pred != c))[0]
        picks += [(i, "correct") for i in ok[np.argsort(-conf[ok])][:per_class]]
        picks += [(i, "wrong") for i in bad[np.argsort(-conf[bad])][:per_class]]
    if not picks: return []
    cols = 4
    rows_n = math.ceil(len(picks) / cols)
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3, rows_n * 3.2))
    axes = np.atleast_1d(axes).ravel()
    exported = []
    for k, (i, kind) in enumerate(picks):
        img = load_batch([frame.path.iloc[i]])
        cam, _ = grad_cam(BEST_MODEL, img)
        ov = overlay(img[0], cam[0])
        axes[k].imshow(ov); axes[k].axis("off")
        axes[k].set_title(f"{'✓' if kind == 'correct' else '✗'} true: {CLASSES[yy[i]]}\npred: {CLASSES[pred[i]]} ({conf[i]:.2f})",
                          fontsize=8, color="green" if kind == "correct" else "red")
        if export_prefix:
            os.makedirs(f"{EXPORT_DIR}/gallery", exist_ok=True)
            fn = f"{export_prefix}_{CLASSES[yy[i]]}_{kind}_{k}.jpg"
            Image.fromarray(ov).save(f"{EXPORT_DIR}/gallery/{fn}", quality=92)
            exported.append(dict(file=fn, true=CLASSES[yy[i]], pred=CLASSES[pred[i]], confidence=float(conf[i]),
                                 kind=kind, source=frame.source.iloc[i]))
    for a in axes[len(picks):]: a.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=160); plt.show()
    return exported

GALLERY = gallery(SPLITS["test"], PRED[BEST]["cal"]["test"], f"Grad-CAM, {BEST}: test set (✓ correct, ✗ wrong)",
                  f"{FIG_DIR}/gradcam_test.png", export_prefix="test")
if "dwsd_test" in PRED[BEST]["cal"]:
    GALLERY += gallery(SPLITS["dwsd_test"], PRED[BEST]["cal"]["dwsd_test"],
                       f"Grad-CAM, {BEST}: Indian held-out crops (DWSD)", f"{FIG_DIR}/gradcam_dwsd.png",
                       per_class=3, export_prefix="dwsd")
json.dump(GALLERY, open(f"{EXPORT_DIR}/gallery.json", "w"), indent=1)

## Step 5 · Export the best model for the Gradio app

In [ ]:
#@title 17 · Export model, settings and example images
r = RESULTS[BEST]
shutil.copy(f"{MODELS_DIR}/{BEST}/infer.keras", f"{EXPORT_DIR}/wastevision.keras")
# Also export a TF SavedModel (used to build the ONNX file for the browser app)
try:
    keras.models.load_model(f"{MODELS_DIR}/{BEST}/infer.keras", compile=False).export(f"{EXPORT_DIR}/saved_model")
except Exception as e:
    print("⚠ SavedModel export failed:", e)
CONFIG = dict(
    version="final", model_file="wastevision.keras", architecture=BEST, img_size=IMG_SIZE,
    preprocessing="RGB -> tf.image.resize_with_pad(224, 224, antialias=True) -> round and clip to 0..255 -> float32. "
                  "No other scaling: the model normalises internally.",
    classes=CLASSES, class_display=CLASS_DISPLAY, streams=STREAMS, stream_of=STREAM_OF,
    sanitary_supported=True, temperature=r["temperature"],
    cost_ratio=COST_RATIO, special_care_threshold=SC_THRESHOLD,
    not_sure_stream_threshold=r["not_sure_stream_threshold"], not_sure_class_threshold=r["not_sure_class_threshold"],
    grad_cam="model.get_layer('preproc') -> get_layer('backbone') (last conv feature map) -> get_layer('head') (logits)",
    metrics=dict(test=r["test"], val=r["val"], test_real=r.get("test_real"), test_web=r.get("test_web"),
                 test_selective_class=r.get("test_selective_class"), test_selective_stream=r.get("test_selective_stream"), dwsd_test=r.get("dwsd_test"), dwsd_ci=r.get("dwsd_ci"),
                 ece_test_after=r["ece_test_after"]),
    data=dict(totals=STATS.get("data_totals"), by_source=STATS.get("data_by_source")),
    created=datetime.datetime.now().isoformat(timespec="seconds"))
json.dump(CONFIG, open(f"{EXPORT_DIR}/config.json", "w"), indent=1)

# Example images for the app (one confident correct test image per class + Indian crops)
ex_dir = f"{EXPORT_DIR}/examples"
shutil.rmtree(ex_dir, ignore_errors=True); os.makedirs(ex_dir)
examples = []
pt, yy, te = PRED[BEST]["cal"]["test"], PRED[BEST]["y"]["test"], SPLITS["test"]
for c in range(NUM_CLASSES):
    ok = np.nonzero((yy == c) & (pt.argmax(1) == c) & te.real.values)[0]
    if not len(ok):
        ok = np.nonzero((yy == c) & (pt.argmax(1) == c))[0]
    if len(ok):
        i = ok[np.argsort(-pt[ok].max(1))][min(2, len(ok) - 1)]
        fn = f"{CLASSES[c]}_{te.source.iloc[i]}.jpg"
        shutil.copy(te.path.iloc[i], f"{ex_dir}/{fn}")
        examples.append(dict(file=fn, label=CLASSES[c], source=te.source.iloc[i]))
if len(SPLITS["dwsd_test"]):
    for k, row in enumerate(SPLITS["dwsd_test"].sample(min(4, len(SPLITS["dwsd_test"])), random_state=SEED).itertuples()):
        fn = f"india_{row.label}_{k}.jpg"
        shutil.copy(row.path, f"{ex_dir}/{fn}")
        examples.append(dict(file=fn, label=row.label, source="DWSD"))
json.dump(examples, open(f"{EXPORT_DIR}/examples.json", "w"), indent=1)
print(f"✓ Exported {BEST} with {len(examples)} example images to {EXPORT_DIR}")

## Step 6 · Research analyses
Leave-one-dataset-out testing, unknown-object hold-outs (batteries, shoes) with MSP / MaxLogit / energy scores, the leakage experiment and the seed summary.

In [ ]:
#@title 18 · Research analyses: leave-one-dataset-out, unknown-object hold-outs, leakage experiment, seeds
# These analyses use frozen ImageNet EfficientNetV2-B0 features with a linear head, so each of the many
# re-trainings takes seconds. The fine-tuned model above is trained on every source, so it cannot be used for
# leave-one-dataset-out testing; the frozen-feature head is the standard, cheap stand-in.
from sklearn.metrics import roc_auc_score
from scipy.special import logsumexp

ANALYSIS = {}
AN_DIR = f"{RESULTS_DIR}/analyses"
os.makedirs(AN_DIR, exist_ok=True)
FEAT_NPZ = f"{DATA_DIR}/frozen_features_effv2b0.npz"

df_all_csv = pd.read_csv(ALL_CSV)
df_all_csv["path"] = NORM_DIR + "/" + df_all_csv.norm_rel
df_all_csv = df_all_csv[df_all_csv.path.map(os.path.exists)].reset_index(drop=True)

def extract_features(paths):
    base = keras.applications.EfficientNetV2B0(include_top=False, weights=PRETRAINED_WEIGHTS, pooling="avg",
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True)
    ds = (tf.data.Dataset.from_tensor_slices(np.asarray(paths, dtype=str)).map(load_image, num_parallel_calls=AUTOTUNE)
          .batch(128).map(lambda x: tf.cast(x, tf.float32)).prefetch(AUTOTUNE))
    f = base.predict(ds, verbose=0)
    del base; keras.backend.clear_session()
    return f.astype(np.float16)

if os.path.exists(FEAT_NPZ):
    z = np.load(FEAT_NPZ, allow_pickle=True)
    FEAT = dict(zip(z["ids"], z["feats"]))
else:
    t0 = time.time()
    feats = extract_features(df_all_csv.path.values)
    np.savez(FEAT_NPZ, ids=df_all_csv.id.values, feats=feats)
    FEAT = dict(zip(df_all_csv.id.values, feats))
    print(f"Frozen features for {len(feats)} images in {(time.time() - t0) / 60:.1f} min")

def X_of(frame):
    return np.stack([FEAT[i] for i in frame.id]).astype(np.float32)

def train_head(X, y, Xv, yv, n_classes, seed=SEED):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([keras.Input((X.shape[1],)), layers.Dropout(0.2), layers.Dense(n_classes)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    cnt = np.bincount(y, minlength=n_classes)
    cw = {int(c): float(len(y) / (np.count_nonzero(cnt) * cnt[c])) for c in np.nonzero(cnt)[0]}
    m.fit(X, y, validation_data=(Xv, yv), epochs=60, batch_size=256, class_weight=cw, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])
    return m

def stream_report(p, y, prefix=""):
    """Class and stream metrics, per-stream recall, the always-Dry baseline and Special Care misses at three cost ratios."""
    out = metrics(p, y)
    s_true, s_pred = CLASS_STREAM_IDX[y], stream_probs(p).argmax(1)
    for k, name in enumerate(STREAMS):
        m = s_true == k
        out[f"recall_{name}"] = float((s_pred[m] == k).mean()) if m.any() else float("nan")
        out[f"n_{name}"] = int(m.sum())
    out["always_dry_baseline"] = float((s_true == STREAMS.index("Dry")).mean())
    for r_ in SENSITIVITY_COST_RATIOS:
        t = 1 / (1 + r_)
        out[f"sc_miss@{t:.2f}"] = 1 - out[f"sc_recall@{t:.2f}"] if not np.isnan(out[f"sc_recall@{t:.2f}"]) else float("nan")
    return {prefix + k: v for k, v in out.items()}

TRAIN = SPLITS["train"][~SPLITS["train"].source.isin(set())]
VAL, TEST = SPLITS["val"], SPLITS["test"]

# ---------- A. Per-stream recall and the always-Dry baseline for every trained model (main test split) ----------
try:
    rows = []
    for arch in RESULTS:
        rows.append(dict(model=arch, **stream_report(PRED[arch]["cal"]["test"], PRED[arch]["y"]["test"])))
        if "dwsd_test" in PRED[arch]["cal"]:
            rows[-1].update(stream_report(PRED[arch]["cal"]["dwsd_test"], PRED[arch]["y"]["dwsd_test"], "dwsd_"))
    per_stream = pd.DataFrame(rows).set_index("model")
    per_stream.to_csv(f"{AN_DIR}/per_stream_recall.csv")
    display(per_stream[[c for c in per_stream.columns if c.startswith(("recall_", "always", "sc_miss", "stream_acc"))]].round(3))
    ANALYSIS["per_stream"] = per_stream.to_dict(orient="index")
except Exception as e:
    print("⚠ per-stream analysis failed:", repr(e)); ANALYSIS["per_stream_error"] = repr(e)

# ---------- B. Seeds: mean ± sd of the main model over MAIN_SEEDS ----------
try:
    seed_runs = [a for a in RESULTS if base_arch(a) == MAIN_ARCH]
    keys = ["class_acc", "macro_f1", "stream_acc", f"sc_recall@{SC_THRESHOLD:.2f}", f"sc_false_alarm@{SC_THRESHOLD:.2f}"]
    rows = []
    for a in seed_runs:
        r = RESULTS[a]
        row = dict(run=a, seed=seed_of(a), **{f"test_{k}": r["test"][k] for k in keys}, ece_test=r["ece_test_after"])
        if "dwsd_test" in r:
            row.update(dwsd_class_acc=r["dwsd_test"]["class_acc"], dwsd_stream_acc=r["dwsd_test"]["stream_acc"])
        rows.append(row)
    seeds = pd.DataFrame(rows).set_index("run")
    summary = seeds.drop(columns="seed").agg(["mean", "std"]).T
    seeds.to_csv(f"{AN_DIR}/seed_runs.csv"); summary.to_csv(f"{AN_DIR}/seed_summary.csv")
    display(summary.round(4))
    ANALYSIS["seeds"] = dict(runs=seeds.to_dict(orient="index"), summary=summary.to_dict(orient="index"))
except Exception as e:
    print("⚠ seed summary failed:", repr(e)); ANALYSIS["seeds_error"] = repr(e)

# ---------- C. Leave-one-dataset-out (LODO): is hazard routing robust to an unseen source? ----------
try:
    all_rows = pd.concat([SPLITS[s] for s in SPLITS], ignore_index=True)
    counts = all_rows.source.value_counts()
    lodo_sources = [s for s, n in counts.items() if n >= LODO_MIN_IMAGES and s not in SYNTH_SOURCES]
    # Reference: one head trained on all training sources, scored on each source's own test images
    Xtr_all, ytr_all = X_of(TRAIN), TRAIN.y.values
    Xva_all, yva_all = X_of(VAL), VAL.y.values
    ref = train_head(Xtr_all, ytr_all, Xva_all, yva_all, NUM_CLASSES)
    T_ref = fit_temperature(ref.predict(Xva_all, verbose=0), yva_all)
    rows = []
    for S in lodo_sources:
        tr, va = TRAIN[TRAIN.source != S], VAL[VAL.source != S]
        te = all_rows[all_rows.source == S]
        missing = sorted(set(te.label) - set(tr.label))
        head = train_head(X_of(tr), tr.y.values, X_of(va), va.y.values, NUM_CLASSES)
        lv = head.predict(X_of(va), verbose=0)
        T = fit_temperature(lv, va.y.values)
        lt = head.predict(X_of(te), verbose=0)
        raw, cal = softmax(lt, 1), softmax(lt / T, 1)
        row = dict(held_out_source=S, n=len(te), classes=", ".join(sorted(te.label.unique())),
                   classes_unseen_in_training=", ".join(missing), temperature=T,
                   ece_before=ece(raw, te.y.values), ece_after=ece(cal, te.y.values),
                   **stream_report(cal, te.y.values))
        te_in = SPLITS["test"][SPLITS["test"].source == S]
        if len(te_in):
            p_in = softmax(ref.predict(X_of(te_in), verbose=0) / T_ref, 1)
            r_in = stream_report(p_in, te_in.y.values, "in_source_")
            row.update({k: r_in[k] for k in ("in_source_n", "in_source_class_acc", "in_source_stream_acc",
                                              f"in_source_sc_recall@{SC_THRESHOLD:.2f}")})
        rows.append(row)
        print(f"  LODO {S:12s} n={len(te):5d}  class acc {row['class_acc']:.3f}  stream acc {row['stream_acc']:.3f}  "
              f"SC miss@0.20 {row.get('sc_miss@0.20', float('nan')):.3f}")
    lodo = pd.DataFrame(rows).set_index("held_out_source")
    lodo.to_csv(f"{AN_DIR}/lodo.csv")
    ANALYSIS["lodo"] = lodo.to_dict(orient="index")
    # Figure: stream accuracy and Special Care miss rate per held-out source, against the in-source reference
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    o = lodo.sort_values("stream_acc")
    axes[0].bar(o.index, o.stream_acc, color="#4C72B0", label="source held out")
    if "in_source_stream_acc" in o:
        axes[0].scatter(o.index, o.in_source_stream_acc, color="black", zorder=3, label="source seen in training")
    axes[0].scatter(o.index, o.always_dry_baseline, marker="_", s=300, color="#C44E52", zorder=3, label="always-Dry baseline")
    axes[0].set_ylabel("Stream accuracy"); axes[0].set_title("Leave-one-dataset-out: SWM stream accuracy"); axes[0].legend(fontsize=8)
    sc = lodo[lodo[f"n_Special Care"] > 0]
    for k, r_ in enumerate(SENSITIVITY_COST_RATIOS):
        t = 1 / (1 + r_)
        axes[1].bar(np.arange(len(sc)) + (k - 1) * 0.27, sc[f"sc_miss@{t:.2f}"], 0.27, label=f"cost {r_:g}:1 (τ={t:.2f})")
    axes[1].set_xticks(range(len(sc))); axes[1].set_xticklabels(sc.index)
    axes[1].set_ylabel("Special Care miss rate"); axes[1].set_title("Hazard misses on an unseen source"); axes[1].legend(fontsize=8)
    for a in axes: a.tick_params(axis="x", rotation=40)
    plt.tight_layout(); plt.savefig(f"{FIG_DIR}/lodo.png", dpi=200); plt.show()
    display(lodo.drop(columns=["classes"]).round(3))
except Exception as e:
    import traceback; traceback.print_exc()
    ANALYSIS["lodo_error"] = repr(e)

# ---------- D. Unknown-object hold-outs: batteries (hazardous) and shoes (harmless) never seen in training ----------
def ood_scores(logits, T):
    return dict(MSP=softmax(logits / T, 1).max(1), MaxLogit=logits.max(1), Energy=T * logsumexp(logits / T, axis=1))

try:
    rows, score_rows = [], []
    for held in HOLDOUT_UNKNOWN:
        keep_cls = [c for c in CLASSES if c != held]
        remap = {CLASS_TO_IDX[c]: i for i, c in enumerate(keep_cls)}
        smat = np.zeros((len(keep_cls), len(STREAMS)), np.float32)
        for i, c in enumerate(keep_cls): smat[i, STREAMS.index(STREAM_OF[c])] = 1
        tr, va = TRAIN[TRAIN.label != held], VAL[VAL.label != held]
        te_id, te_ood = TEST[TEST.label != held], TEST[TEST.label == held]
        head = train_head(X_of(tr), tr.y.map(remap).values, X_of(va), va.y.map(remap).values, len(keep_cls))
        lv, lid, lood = (head.predict(X_of(f), verbose=0) for f in (va, te_id, te_ood))
        T = fit_temperature(lv, va.y.map(remap).values)
        sv, sid, sood = ood_scores(lv, T), ood_scores(lid, T), ood_scores(lood, T)
        p_ood, p_id = softmax(lood / T, 1), softmax(lid / T, 1)
        thr = float(np.quantile(sv["MSP"], 0.05))   # rejects 5% of known validation items
        rej_ood, rej_id = sood["MSP"] < thr, sid["MSP"] < thr
        st_ood = p_ood @ smat
        sc_flag = st_ood[:, SC] >= SC_THRESHOLD
        top_stream = np.array(STREAMS)[st_ood.argmax(1)]
        row = dict(held_out_class=held, true_stream=STREAM_OF[held], n_unknown=len(te_ood), msp_reject_threshold=thr,
                   known_items_rejected=float(rej_id.mean()),
                   unknown_rejected=float(rej_ood.mean()),
                   unknown_flagged_special_care=float(sc_flag.mean()),
                   unknown_rejected_or_flagged=float((rej_ood | sc_flag).mean()),
                   unknown_silently_routed_wet=float((~rej_ood & ~sc_flag & (top_stream == "Wet")).mean()),
                   unknown_silently_routed_dry=float((~rej_ood & ~sc_flag & (top_stream == "Dry")).mean()),
                   unknown_silently_routed_sanitary=float((~rej_ood & ~sc_flag & (top_stream == "Sanitary")).mean()),
                   most_common_predicted_class=pd.Series(np.array(keep_cls)[p_ood.argmax(1)]).value_counts().head(3).to_dict())
        for name in ("MSP", "MaxLogit", "Energy"):
            yb = np.r_[np.ones(len(lid)), np.zeros(len(lood))]
            row[f"auroc_{name}"] = float(roc_auc_score(yb, np.r_[sid[name], sood[name]]))
            # false-positive rate on unknowns at 95% true-positive rate on known items
            t95 = np.quantile(sid[name], 0.05)
            row[f"fpr95_{name}"] = float((sood[name] >= t95).mean())
        rows.append(row)
        print(f"  hold-out {held}: rejected {row['unknown_rejected']:.2f} | flagged Special Care {row['unknown_flagged_special_care']:.2f} "
              f"| AUROC MSP {row['auroc_MSP']:.3f} MaxLogit {row['auroc_MaxLogit']:.3f} Energy {row['auroc_Energy']:.3f}")
    holdout = pd.DataFrame(rows).set_index("held_out_class")
    holdout.to_csv(f"{AN_DIR}/holdout_unknown.csv")
    ANALYSIS["holdout_unknown"] = holdout.to_dict(orient="index")
    display(holdout.T)
except Exception as e:
    import traceback; traceback.print_exc()
    ANALYSIS["holdout_error"] = repr(e)

# ---------- E. Leakage experiment: naive random split vs duplicate-aware grouped split on the same images ----------
try:
    pool = df_all_csv[(df_all_csv.source != "DWSD") & ~df_all_csv.source.isin(SYNTH_SOURCES)].reset_index(drop=True)
    pool["y"] = pool.label.map(CLASS_TO_IDX).astype(int)
    # duplicate clusters at the dedup threshold, merged with the parent-photo groups
    pr = pd.read_csv(PAIRS_CSV)
    pr = pr[pr.distance <= PHASH_DUP_THRESHOLD]
    pos = {i: k for k, i in enumerate(pool.id)}
    uf = UnionFind(len(pool))
    for a, b in zip(pr.id_a, pr.id_b):
        if a in pos and b in pos: uf.union(pos[a], pos[b])
    gid = pool.group.astype(str).values
    first = {}
    for k, g in enumerate(gid):
        if g in first: uf.union(first[g], k)
        else: first[g] = k
    pool["dup_group"] = [uf.find(k) for k in range(len(pool))]
    rng = np.random.default_rng(SEED)
    Xp, yp = X_of(pool), pool.y.values
    H = np.array([int(h, 16) for h in pool.phash], dtype=np.uint64)
    rows = []
    for protocol in ("naive random split (no de-duplication, no grouping)", "grouped by photo and duplicate cluster"):
        if protocol.startswith("naive"):
            r_ = rng.random(len(pool)); test_m = r_ < 0.2; val_m = (r_ >= 0.2) & (r_ < 0.3)
        else:
            ug = pool.dup_group.unique(); rr = dict(zip(ug, rng.random(len(ug))))
            r_ = pool.dup_group.map(rr).values; test_m = r_ < 0.2; val_m = (r_ >= 0.2) & (r_ < 0.3)
        tr_m = ~test_m & ~val_m
        head = train_head(Xp[tr_m], yp[tr_m], Xp[val_m], yp[val_m], NUM_CLASSES)
        T = fit_temperature(head.predict(Xp[val_m], verbose=0), yp[val_m])
        p = softmax(head.predict(Xp[test_m], verbose=0) / T, 1)
        # which test images have a near-duplicate in this protocol's training set?
        Htr, Hte = H[tr_m], H[test_m]
        near = np.zeros(len(Hte), bool)
        for a in range(0, len(Hte), 256):
            near[a:a + 256] = popcount64(Hte[a:a + 256, None] ^ Htr[None, :]).min(1) <= PHASH_DUP_THRESHOLD
        m = metrics(p, yp[test_m])
        pred_ok = p.argmax(1) == yp[test_m]
        rows.append(dict(protocol=protocol, n_test=int(test_m.sum()), class_acc=m["class_acc"], macro_f1=m["macro_f1"],
                         stream_acc=m["stream_acc"], test_with_near_duplicate_in_train=float(near.mean()),
                         acc_items_with_duplicate=float(pred_ok[near].mean()) if near.any() else float("nan"),
                         acc_items_without_duplicate=float(pred_ok[~near].mean()) if (~near).any() else float("nan")))
    leak = pd.DataFrame(rows).set_index("protocol")
    leak.to_csv(f"{AN_DIR}/leakage_experiment.csv")
    ANALYSIS["leakage_experiment"] = leak.to_dict(orient="index")
    display(leak.round(4))
except Exception as e:
    import traceback; traceback.print_exc()
    ANALYSIS["leakage_error"] = repr(e)

json.dump(ANALYSIS, open(f"{AN_DIR}/research_analyses.json", "w"), indent=1,
          default=lambda o: o.item() if hasattr(o, "item") else str(o))
print(f"✓ Research analyses saved to {AN_DIR}")


In [ ]:
#@title 19 · Summary for the report
def md_table(df):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join([df.index.name or ""] + cols) + " |", "|" + "---|" * (len(cols) + 1)]
    for idx, row in df.iterrows():
        lines.append("| " + " | ".join([str(idx)] + [str(v) for v in row.values]) + " |")
    return "\n".join(lines)

r = RESULTS[BEST]
lines = [f"# WasteVision results ({datetime.date.today()})", "",
         f"Environment: {STATS.get('environment')}", "",
         "## Data", f"Sources downloaded: {STATS.get('sources_ok')}; skipped: {STATS.get('sources_failed')}",
         f"Totals by split: {STATS.get('data_totals')}", f"By source: {STATS.get('data_by_source')}",
         f"Mapped images per source and class (before dedup/cap): {STATS.get('mapped_counts')}",
         f"Share of real photos per split: {STATS.get('real_photo_share')}",
         f"Raw images found: {STATS.get('raw_counts')}", f"Excluded folders: {STATS.get('excluded_folders')}",
         f"Cleaning drops: {STATS.get('cleaning_dropped')}", f"Deduplication: {STATS.get('dedup')}",
         f"Leakage audit (held-out near-duplicates of train): {STATS.get('leakage_audit_near_duplicates_vs_train')}",
         f"DWSD: files {STATS.get('dwsd_files')}, mask formats {STATS.get('dwsd_mask_kinds')}, "
         f"mapping {STATS.get('dwsd_mask_mapping', 'index values')}, photos {STATS.get('dwsd_parent_photos')}, "
         f"groups {STATS.get('dwsd_parent_groups')}, crops by class {STATS.get('dwsd_crops_by_class')}, "
         f"skipped {STATS.get('dwsd_crops_skipped')}, zip checksum ok: {STATS.get('dwsd_zip_sha256_matches')}", "",
         "## Model comparison", md_table(COMPARISON), "",
         f"## Best model: {BEST} (selected on validation macro-F1)",
         f"Temperature: {r['temperature']:.3f}; ECE val {r['ece_val_before']:.3f} -> {r['ece_val_after']:.3f}; "
         f"ECE test {r['ece_test_before']:.3f} -> {r['ece_test_after']:.3f}",
         f"Test metrics: {json.dumps(r['test'])}",
         f"Validation metrics: {json.dumps(r['val'])}",
         f"Not-sure thresholds (from validation): stream {r['not_sure_stream_threshold']:.3f}, class {r['not_sure_class_threshold']:.3f}; "
         f"targets met: {r['not_sure_targets_met_on_val']}",
         f"Test selective stream: {r['test_selective_stream']}; selective class: {r['test_selective_class']}",
         f"Test accuracy by source: {r['test_by_source']}",
         f"Test, real photos only: {json.dumps(r.get('test_real'))}",
         f"Test, web/mixed photos only: {json.dumps(r.get('test_web'))}",
         f"Indian (DWSD) held-out: {json.dumps(r.get('dwsd_test'))}; 95% CIs: {r.get('dwsd_ci')}; groups: {r.get('dwsd_groups')}",
         "", "## Research analyses (frozen-feature heads)", "```json",
         json.dumps(ANALYSIS if "ANALYSIS" in globals() else {}, indent=1, default=str), "```",
         "", "## Per-class report (test)", "```", report, "```",
         "", "## All models (full numbers)", "```json", json.dumps(RESULTS, indent=1), "```"]
open(f"{RESULTS_DIR}/summary.md", "w").write("\n".join(lines))
shutil.copy(f"{EXPORT_DIR}/config.json", f"{RESULTS_DIR}/export_config.json")
zip_base = f"{PROJECT_DIR}/results"
shutil.make_archive(zip_base, "zip", RESULTS_DIR)
display(Markdown(f"**Done.** Download `{zip_base}.zip` from Google Drive for the report."))